# BRCA1 ESM-DMS Experimental Pipeline

This notebook runs the class-based BRCA1 workflow from `data/mavedb_data`: MaveDB counts and functional scores are keyed by `hgvs_nt`, mutated protein sequences are reconstructed from the BRCA1 wildtype protein reference where the nucleotide mutation has an unambiguous amino-acid consequence, embeddings are pooled and cached, DeltaSAE features are inferred, and model fitness is compared with MaveDB functional scores.

In [ ]:
from pathlib import Path

In [ ]:
import os
import json
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "esmDMS.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "esmDMS.py").exists():
            REPO_ROOT = parent
            break

os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from esmDMS import CellularDMSInput, ESMDMSConfig, esmDMS

DATA_DIR = REPO_ROOT / "data" / "mavedb_data"
ANALYSIS_DIR = REPO_ROOT / "data" / "esm_data_analysis" / "BRCA1_experimental"
SEQUENCE_DIR = ANALYSIS_DIR / "sequence_data"
FIGURE_DIR = ANALYSIS_DIR / "figures"
TABLE_DIR = ANALYSIS_DIR / "tables"

for directory in (SEQUENCE_DIR, FIGURE_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="darkgrid")
REPO_ROOT

## Configure BRCA1

`SAE_EMBEDDING_TYPE` selects which saved pooled embedding cache trains or feeds the sparse autoencoder. Use `"mean_pool"` or `"max_pool"`. `EMBEDDING_MODEL` can be an ESM-2 Hugging Face model such as `"facebook/esm2_t6_8M_UR50D"` or an ESMC model identifier supported by the local environment.

In [ ]:
BRCA1_INPUT = CellularDMSInput(
    reference_nuc_path=DATA_DIR / "BRCA1_reference_sequence.dat",
    mavedb_csv_path=DATA_DIR / "BRCA1_counts.csv",
    scores_csv_path=DATA_DIR / "BRCA1_scores.csv",
    reference_kind="protein",
    primary_key="hgvs_nt",
)

EMBEDDING_MODEL = "biohub/ESMC-300M"
REPRESENTATIVE_LAYER = 23
SAE_EMBEDDING_TYPE = "max_pool"  # "mean_pool" or "max_pool"
ABSTRACTION_METHOD = "DeltaSAE"
NORM_SCHEME = "none"

SAE_PARAMS = {
    "n_features": 10000,
    "sparsity_coeff": 1e-3,
    "sparsity_mode": "batchtopk",  # "normal", "topk", or "batchtopk"
    "k": 8,
    "epochs": 200,
    "batch_size": 64,
    "train_frac": 0.8,
    "lr": 1e-3,
    "seed": 42,
    "run_label": f"{SAE_EMBEDDING_TYPE}_batchtopk200_12800feat",
    "norm_scheme": NORM_SCHEME,
    # "pretrained_model_path": SEQUENCE_DIR / "sae_models" / "existing_model.pt",
}

config = ESMDMSConfig(
    embedding_model=EMBEDDING_MODEL,
    embedding_type=SAE_EMBEDDING_TYPE,
    local_or_disk="both",
    save_dir=str(SEQUENCE_DIR),
    dataset_name="BRCA1",
)

runner = esmDMS(input_data=BRCA1_INPUT, config=config)
runner

## Process Counts And Scores

The parser keeps `hgvs_nt` as `SequenceIndex`, stores a separate key-to-protein-sequence map, loads functional scores keyed by `hgvs_nt`, and skips rows that cannot produce a unique protein sequence from a protein-only reference, such as intronic HGVS entries or ambiguous codon effects.

In [ ]:
runner.process_raw_data(drop_stop_codons=True)

processing_summary = pd.DataFrame([{
    "dataset": "BRCA1",
    "reference_kind": runner.reference_kind,
    "count_rows": len(runner.sequence_dataframe),
    "mutation_keys_in_counts": runner.sequence_dataframe["SequenceIndex"].nunique(),
    "protein_sequence_keys": len(runner.sequence_to_protein_sequence),
    "replicates": runner.sequence_dataframe["Replicate"].nunique(),
    "generations": sorted(runner.sequence_dataframe["Generation"].unique()),
    "score_rows": len(runner.scores_dataframe),
    "skipped_counts": runner.sequence_metadata.attrs.get("skipped_counts", {}),
}])
processing_summary.to_csv(TABLE_DIR / "BRCA1_processing_summary.csv", index=False)
processing_summary

In [ ]:
ANNOTATION_PATH = DATA_DIR / "BRCA1_annotations.ndjson"
ANNOTATION_LABELS = {"benign", "pathogenic", "uncertain significance"}


def _load_brca1_pathogenicity_annotations(annotation_path, score_path, primary_key):
    records = []
    with Path(annotation_path).open() as f:
        for line in f:
            record = json.loads(line)
            annotation = record.get("annotation") or {}
            classification = annotation.get("classification") or {}
            coding = classification.get("primaryCoding") or {}
            annotation_label = coding.get("code")
            if annotation_label not in ANNOTATION_LABELS:
                continue
            records.append({
                "annotation_identifier": record["variant_urn"],
                "annotation": annotation_label,
            })

    annotation_df = pd.DataFrame(records).drop_duplicates("annotation_identifier")
    key_df = pd.read_csv(score_path, usecols=["accession", primary_key]).rename(
        columns={"accession": "annotation_identifier"}
    )
    annotation_df = key_df.merge(annotation_df, on="annotation_identifier", how="inner")
    annotation_df = annotation_df.dropna(subset=[primary_key, "annotation"])
    return annotation_df


annotation_df = _load_brca1_pathogenicity_annotations(
    ANNOTATION_PATH,
    BRCA1_INPUT.scores_csv_path,
    BRCA1_INPUT.primary_key,
)
runner.annotation_dataframe = annotation_df
runner.primary_key_to_annotation_identifier = dict(zip(
    annotation_df[BRCA1_INPUT.primary_key],
    annotation_df["annotation_identifier"],
))
runner.primary_key_to_annotation = dict(zip(
    annotation_df[BRCA1_INPUT.primary_key],
    annotation_df["annotation"],
))

if runner.scores_dataframe is not None:
    runner.scores_dataframe = runner.scores_dataframe.drop(
        columns=["annotation_identifier", "annotation"],
        errors="ignore",
    ).merge(
        annotation_df[[BRCA1_INPUT.primary_key, "annotation_identifier", "annotation"]],
        on=BRCA1_INPUT.primary_key,
        how="left",
    )

annotation_summary = (
    annotation_df["annotation"]
    .value_counts()
    .rename_axis("annotation")
    .reset_index(name="n_variants")
)
annotation_summary.to_csv(TABLE_DIR / "BRCA1_annotation_summary.csv", index=False)
annotation_summary


In [ ]:
runner.sequence_dataframe.head()

In [ ]:
runner.scores_dataframe.head()

## Embed Mutated And Wildtype Proteins

Embedding writes `mean_pool`, `max_pool`, and `per_residue` caches for each layer. The wildtype protein is embedded even though it does not appear as a count row, because DeltaSAE subtracts its SAE representation.

In [ ]:
RUN_LOCAL_EMBEDDINGS = False
CREATE_EMBEDDING_JOB = False
SUBMIT_JOBS = False

if RUN_LOCAL_EMBEDDINGS:
    runner.embed_all_sequences(layer=REPRESENTATIVE_LAYER, test_num=10)

if CREATE_EMBEDDING_JOB:
    embedding_job = runner.create_embedding_batch_job(
        job_dir=SEQUENCE_DIR / "embedding_batch_jobs",
        n_chunks=40,
        max_active_jobs=4,
        job_name="brca1_esm_embed",
        partition="any_cpu",
        mem="24G",
        time="08:00:00",
        python_executable="python3",
        scratch_root="/scr",
        submit=SUBMIT_JOBS,
    )
    display(pd.DataFrame([{
        "script_path": str(embedding_job["script_path"]),
        "payload_path": str(embedding_job["payload_path"]),
        "job_id": embedding_job["job_id"],
    }]))

In [ ]:
MERGE_EMBEDDING_OUTPUTS = False

if MERGE_EMBEDDING_OUTPUTS:
    runner.merge_embedding_batch_outputs(
        job_dir=SEQUENCE_DIR / "embedding_batch_jobs",
        layer="all",
        save_layers=True,
    )

cache_status = pd.DataFrame([
    {
        "layer": REPRESENTATIVE_LAYER,
        "embedding_type": embedding_type,
        "path": str(runner._embedding_path(REPRESENTATIVE_LAYER, embedding_type)),
        "exists": runner._embedding_path(REPRESENTATIVE_LAYER, embedding_type).exists(),
    }
    for embedding_type in ("mean_pool", "max_pool", "per_residue")
])
cache_status.to_csv(TABLE_DIR / "BRCA1_embedding_cache_status.csv", index=False)
cache_status

## CPU DeltaSAE Architecture Sweep

Create Slurm CPU jobs for grids of DeltaSAE and DeltaEmbSAE architectures. Each job handles one method and one pooling type, and `SAE_SWEEP_MAX_PARALLEL_RUNS` controls how many SAE configurations train concurrently inside each allocation.


In [ ]:
SAE_SWEEP_BASE_PARAMS = {
    "sparsity_coeff": 1e-3,
    "sparsity_mode": "batchtopk",
    "epochs": 200,
    "batch_size": 64,
    "train_frac": 0.8,
    "lr": 1e-3,
    "norm_scheme": NORM_SCHEME,
}
SAE_SWEEP_N_FEATURES = [8192, 12800, 16384]
SAE_SWEEP_K_VALUES = [64, 128, 256, 512]
SAE_SWEEP_EMBEDDING_TYPES = ["mean_pool", "max_pool"]
SAE_METHODS = ["DeltaSAE", "DeltaEmbSAE"]
SAE_SWEEP_SEEDS = [42]

SAE_SWEEP_PARAM_GRID = []
for n_features in SAE_SWEEP_N_FEATURES:
    for k in SAE_SWEEP_K_VALUES:
        for seed in SAE_SWEEP_SEEDS:
            SAE_SWEEP_PARAM_GRID.append({
                **SAE_SWEEP_BASE_PARAMS,
                "n_features": n_features,
                "k": k,
                "seed": seed,
            })

CREATE_SAE_SWEEP_JOB = True
SUBMIT_SAE_SWEEP_JOB = False
SAE_SWEEP_CPUS_PER_RUN = 4
SAE_SWEEP_MAX_PARALLEL_RUNS = 4
SAE_SWEEP_CPUS_PER_TASK = SAE_SWEEP_CPUS_PER_RUN * SAE_SWEEP_MAX_PARALLEL_RUNS

sae_sweep_jobs = []
if CREATE_SAE_SWEEP_JOB:
    for method in SAE_METHODS:
        for embedding_type in SAE_SWEEP_EMBEDDING_TYPES:
            sweep_params = []
            for params in SAE_SWEEP_PARAM_GRID:
                run_params = dict(params)
                run_params["run_label"] = (
                    f"{method}_{embedding_type}_batchtopk_k{run_params['k']}_"
                    f"nf{run_params['n_features']}_seed{run_params['seed']}"
                )
                sweep_params.append(run_params)

            sae_sweep_dir = SEQUENCE_DIR / "sae_sweep_jobs_new" / (
                f"BRCA1_{embedding_type}_{method}_L{REPRESENTATIVE_LAYER}_batchtopk_grid_cpu"
            )
            sae_sweep_job = runner.create_sae_sweep_job(
                layer=REPRESENTATIVE_LAYER,
                method=method,
                embedding_type=embedding_type,
                sweep_params=sweep_params,
                job_dir=sae_sweep_dir,
                job_name=f"brca1_{method.lower()}_{embedding_type}_sweep_cpu",
                partition="any_cpu",
                gpus=1,
                gres="",
                cpus_per_task=SAE_SWEEP_CPUS_PER_TASK,
                mem="16G",
                time="4:00:00",
                max_parallel_runs=SAE_SWEEP_MAX_PARALLEL_RUNS,
                run_inference=True,
                force_recompute=False,
                require_cuda=False,
                submit=SUBMIT_SAE_SWEEP_JOB,
            )
            sae_sweep_job.update({"method": method, "embedding_type": embedding_type})
            sae_sweep_jobs.append(sae_sweep_job)

    display(pd.DataFrame([
        {
            "method": job["method"],
            "embedding_type": job["embedding_type"],
            "n_sweep_configs": len(SAE_SWEEP_PARAM_GRID),
            "sweep_dir": job["sweep_dir"],
            "script_path": job["script_path"],
            "job_id": job["job_id"],
            "submitted": SUBMIT_SAE_SWEEP_JOB,
        }
        for job in sae_sweep_jobs
    ]))


## Plain SAE CPU Sweep

Submit the same architecture grid using regular SAE features directly on pooled embeddings. This does not subtract wildtype embeddings or wildtype SAE activations.


In [ ]:
SAE_ONLY_METHOD = "SAE"
CREATE_SAE_ONLY_SWEEP_JOB = True
SUBMIT_SAE_ONLY_SWEEP_JOB = False

sae_only_sweep_jobs = []
if CREATE_SAE_ONLY_SWEEP_JOB:
    for embedding_type in SAE_SWEEP_EMBEDDING_TYPES:
        sweep_params = []
        for params in SAE_SWEEP_PARAM_GRID:
            run_params = dict(params)
            run_params["run_label"] = (
                f"{SAE_ONLY_METHOD}_{embedding_type}_batchtopk_k{run_params['k']}_"
                f"nf{run_params['n_features']}_seed{run_params['seed']}"
            )
            sweep_params.append(run_params)

        sae_only_sweep_dir = SEQUENCE_DIR / "sae_sweep_jobs_new" / (
            f"BRCA1_{embedding_type}_{SAE_ONLY_METHOD}_L{REPRESENTATIVE_LAYER}_batchtopk_grid_cpu"
        )
        sae_only_sweep_job = runner.create_sae_sweep_job(
            layer=REPRESENTATIVE_LAYER,
            method=SAE_ONLY_METHOD,
            embedding_type=embedding_type,
            sweep_params=sweep_params,
            job_dir=sae_only_sweep_dir,
            job_name=f"brca1_{SAE_ONLY_METHOD.lower()}_{embedding_type}_sweep_cpu",
            partition="any_cpu",
            gpus=1,
            gres="",
            cpus_per_task=SAE_SWEEP_CPUS_PER_TASK,
            mem="16G",
            time="4:00:00",
            max_parallel_runs=SAE_SWEEP_MAX_PARALLEL_RUNS,
            run_inference=True,
            force_recompute=False,
            require_cuda=False,
            submit=SUBMIT_SAE_ONLY_SWEEP_JOB,
        )
        sae_only_sweep_job.update({"method": SAE_ONLY_METHOD, "embedding_type": embedding_type})
        sae_only_sweep_jobs.append(sae_only_sweep_job)

    display(pd.DataFrame([
        {
            "method": job["method"],
            "embedding_type": job["embedding_type"],
            "n_sweep_configs": len(SAE_SWEEP_PARAM_GRID),
            "sweep_dir": job["sweep_dir"],
            "script_path": job["script_path"],
            "job_id": job["job_id"],
            "submitted": SUBMIT_SAE_ONLY_SWEEP_JOB,
        }
        for job in sae_only_sweep_jobs
    ]))


## DeltaSAE Sweep Pareto Fronts

Load completed sweep outputs and compare activation sparsity, SAE reconstruction accuracy, and MaveDB Spearman rho. The Pareto fronts treat both axes as objectives to maximize. Exact regular popDMS inference is kept in the separate cache-generation cell below so plotting cells do not launch `paperPop.infer_correlated` implicitly.


In [ ]:
import itertools
import pickle
import numpy as np
from scipy.stats import pearsonr, spearmanr
from popDMS import infer_gamma_range, mini_infer_esm
import paperPop

SAE_SWEEP_ANALYSIS_DIR = SEQUENCE_DIR / "sae_sweep_jobs_new"
SAE_SWEEP_RESULTS_PATHS = sorted(SAE_SWEEP_ANALYSIS_DIR.glob("*/sae_sweep_results.csv"))
RUN_SAE_SWEEP_PARETO_ANALYSIS = False
RUN_RAW_EMBEDDING_BENCHMARKS = True
RUN_EXACT_POPDMS_BASELINE = True
FORCE_RECOMPUTE_EXACT_POPDMS_BASELINE = False
PLOT_GAMMA_REGULARIZATION_FOR_ALL_MODELS = False
PLOT_POOL_ONLY_PARETO = False
PLOT_METHOD_POOL_PARETO = False
GAMMA_REGULARIZATION_VALUES = None
RAW_EMBEDDING_BENCHMARK_DIR = SEQUENCE_DIR / "benchmark_inference"
REGULAR_POPDMS_DIR = SEQUENCE_DIR / "regular_popdms"
EXACT_POPDMS_NAME = "BRCA1_exact_popDMS"
EXACT_POPDMS_GAMMA = None
EXACT_POPDMS_CORR_CUTOFF_PCT = 0.5
EXACT_POPDMS_NORM_WT = False
EXACT_POPDMS_SELECTION_PATH = REGULAR_POPDMS_DIR / f"{EXACT_POPDMS_NAME}_selection_coefficients.csv.gz"
EXACT_POPDMS_SELECTION_TABLE_PATH = TABLE_DIR / "BRCA1_exact_popDMS_selection_coefficients.csv"
EXACT_POPDMS_FITNESS_PATH = TABLE_DIR / "BRCA1_exact_popDMS_fitness_values.csv"
FIGURE_DPI = 450
R2_AXIS_EPSILON = 0.02
PLOT_AX_SIZE = 5.4
LEGEND_WIDTH = 3.2
PLOT_USE_TEX = False

plt.rcParams.update({
    "text.usetex": PLOT_USE_TEX,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 8,
    "legend.title_fontsize": 9,
})


def _square_axes_figure(ax_size=PLOT_AX_SIZE, legend_width=LEGEND_WIDTH):
    left = 0.95
    bottom = 0.75
    top = 0.45
    legend_gap = 0.30
    right = 0.15
    fig_width = left + ax_size + legend_gap + legend_width + right
    fig_height = bottom + ax_size + top
    fig = plt.figure(figsize=(fig_width, fig_height))
    ax = fig.add_axes([
        left / fig_width,
        bottom / fig_height,
        ax_size / fig_width,
        ax_size / fig_height,
    ])
    legend_anchor = ((left + ax_size + legend_gap) / fig_width, (bottom + ax_size) / fig_height)
    return fig, ax, legend_anchor


def _latex_safe_label(value):
    label = str(value)
    if not PLOT_USE_TEX:
        return label
    return label.replace("_", r"\_")


def _external_legend(fig, ax, legend_anchor, handles=None, labels=None, **kwargs):
    if handles is None or labels is None:
        handles, labels = ax.get_legend_handles_labels()
    labels = [_latex_safe_label(label) for label in labels]
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()
    if handles:
        fig.legend(
            handles,
            labels,
            loc="upper left",
            bbox_to_anchor=legend_anchor,
            frameon=False,
            borderaxespad=0.0,
            **kwargs,
        )


def _load_pickle(path):
    with Path(path).open("rb") as f:
        return pickle.load(f)


def _as_existing_path(value):
    if value is None or pd.isna(value):
        return None
    path = Path(value)
    return path if path.is_file() else None


def _reconstruction_metrics(viz_path):
    viz_path = _as_existing_path(viz_path)
    if viz_path is None:
        return {
            "reconstruction_mse": np.nan,
            "reconstruction_r2": np.nan,
            "activation_sparsity": np.nan,
            "activation_density": np.nan,
            "n_active_features": np.nan,
            "active_feature_fraction": np.nan,
            "final_train_loss": np.nan,
            "final_test_loss": np.nan,
        }
    viz = _load_pickle(viz_path)
    X = np.asarray(viz["X_original"], dtype=float)
    X_recon = np.asarray(viz["X_reconstructed"], dtype=float)
    test_idx = np.asarray(viz.get("test_idx") or [], dtype=int)
    eval_idx = test_idx if len(test_idx) else np.arange(X.shape[0])
    X_eval = X[eval_idx]
    X_recon_eval = X_recon[eval_idx]
    residual = X_eval - X_recon_eval
    mse = float(np.mean(residual ** 2))
    sse = float(np.sum(residual ** 2))
    centered = X_eval - X_eval.mean(axis=0, keepdims=True)
    sst = float(np.sum(centered ** 2))
    r2 = 1.0 - (sse / sst) if sst > 0 else np.nan

    Z_all = np.asarray(viz["Z_all"])
    activation_density = float((Z_all > 0).mean())
    active_mask = np.asarray(viz["active_mask"], dtype=bool)
    train_losses = viz.get("train_losses") or []
    test_losses = viz.get("test_losses") or []
    return {
        "reconstruction_mse": mse,
        "reconstruction_r2": r2,
        "activation_sparsity": 1.0 - activation_density,
        "activation_density": activation_density,
        "n_active_features": int(active_mask.sum()),
        "active_feature_fraction": float(active_mask.mean()),
        "final_train_loss": float(train_losses[-1]) if train_losses else np.nan,
        "final_test_loss": float(test_losses[-1]) if test_losses else np.nan,
    }


def _spearman_rho_for_run(feature_path, inference_path, norm_scheme=NORM_SCHEME, score_col="score"):
    feature_path = _as_existing_path(feature_path)
    inference_path = _as_existing_path(inference_path)
    if feature_path is None or inference_path is None:
        return np.nan, 0
    if runner.scores_dataframe is None:
        runner.load_functional_scores()
    score_df = runner.scores_dataframe[["SequenceIndex", score_col]].dropna().copy()
    score_map = dict(zip(score_df["SequenceIndex"], score_df[score_col]))
    seq_to_features = _load_pickle(feature_path)
    inference_result = _load_pickle(inference_path)
    seq_ids = [seq_id for seq_id, feature in seq_to_features.items() if seq_id in score_map and feature is not None]
    if not seq_ids:
        return np.nan, 0
    features = np.asarray([seq_to_features[seq_id] for seq_id in seq_ids], dtype=float)
    if norm_scheme is not None and norm_scheme != "none":
        features = esmDMS._normalize_features(features, norm_scheme)
    fitness = 1.0 + features @ inference_result.s_joint
    scores = np.asarray([score_map[seq_id] for seq_id in seq_ids], dtype=float)
    finite = np.isfinite(fitness) & np.isfinite(scores)
    if finite.sum() < 3:
        return np.nan, int(finite.sum())
    return float(spearmanr(fitness[finite], scores[finite]).statistic), int(finite.sum())


def _spearman_rho_for_features(seq_to_features, inference_result, norm_scheme=NORM_SCHEME, score_col="score"):
    if runner.scores_dataframe is None:
        runner.load_functional_scores()
    score_df = runner.scores_dataframe[["SequenceIndex", score_col]].dropna().copy()
    score_map = dict(zip(score_df["SequenceIndex"], score_df[score_col]))
    seq_ids = [seq_id for seq_id, feature in seq_to_features.items() if seq_id in score_map and feature is not None]
    if not seq_ids:
        return np.nan, 0
    features = np.asarray([seq_to_features[seq_id] for seq_id in seq_ids], dtype=float)
    if norm_scheme is not None and norm_scheme != "none":
        features = esmDMS._normalize_features(features, norm_scheme)
    fitness = 1.0 + features @ inference_result.s_joint
    scores = np.asarray([score_map[seq_id] for seq_id in seq_ids], dtype=float)
    finite = np.isfinite(fitness) & np.isfinite(scores)
    if finite.sum() < 3:
        return np.nan, int(finite.sum())
    return float(spearmanr(fitness[finite], scores[finite]).statistic), int(finite.sum())


def _raw_delta_embedding_features(layer, embedding_type):
    embeddings = runner.load_embeddings(layer, embedding_type)
    wildtype_key = getattr(runner.input_data, "wildtype_key", None)
    if wildtype_key is None or wildtype_key not in embeddings:
        raise KeyError(f"Wildtype key {wildtype_key!r} is not present in {embedding_type} embeddings.")
    wt_embedding = np.asarray(embeddings[wildtype_key], dtype=np.float32)
    return {
        seq_id: np.asarray(embedding, dtype=np.float32) - wt_embedding
        for seq_id, embedding in embeddings.items()
        if seq_id != wildtype_key
    }


def _raw_delta_embedding_inference(layer, embedding_type, norm_scheme=NORM_SCHEME):
    RAW_EMBEDDING_BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
    layer_label = esmDMS._layer_label(layer)
    inference_path = RAW_EMBEDDING_BENCHMARK_DIR / (
        f"BRCA1_{embedding_type}_raw_delta_embeddings_{layer_label}_{norm_scheme}_inference_results.pkl"
    )
    feature_path = RAW_EMBEDDING_BENCHMARK_DIR / (
        f"BRCA1_{embedding_type}_raw_delta_embeddings_{layer_label}_seq_to_features.pkl"
    )
    seq_to_features = _raw_delta_embedding_features(layer, embedding_type)
    if not feature_path.is_file():
        esmDMS._save_pickle(seq_to_features, feature_path)
    if inference_path.is_file():
        inference_result = _load_pickle(inference_path)
    else:
        inference_features = seq_to_features
        if norm_scheme is not None and norm_scheme != "none":
            seq_ids = list(inference_features)
            features = np.asarray([inference_features[seq_id] for seq_id in seq_ids], dtype=float)
            features = esmDMS._normalize_features(features, norm_scheme)
            inference_features = dict(zip(seq_ids, features))
        sequence_dataframe, inference_features = esmDMS._drop_missing_features(
            runner.sequence_dataframe,
            inference_features,
            "Raw delta embedding benchmark",
        )
        esmDMS._require_vector_features(inference_features, "Raw delta embedding benchmark")
        inference_result = mini_infer_esm(sequence_dataframe, inference_features)
        esmDMS._save_pickle(inference_result, inference_path)
    return seq_to_features, inference_result, feature_path, inference_path


def _raw_embedding_benchmark_rows(embedding_types, layer, norm_scheme=NORM_SCHEME):
    rows = []
    for embedding_type in embedding_types:
        raw_features = runner.load_embeddings(layer, embedding_type)
        raw_inference = runner.run_feature_inference(
            layer=layer,
            abstraction_method="none",
            abstraction_params={"norm_scheme": norm_scheme},
            embedding_type=embedding_type,
        )
        raw_rho, raw_n = _spearman_rho_for_features(raw_features, raw_inference, norm_scheme=norm_scheme)
        rows.append({
            "benchmark": "Raw embeddings",
            "embedding_type": embedding_type,
            "spearman_rho": raw_rho,
            "n_score_sequences": raw_n,
        })

        delta_features, delta_inference, _, _ = _raw_delta_embedding_inference(layer, embedding_type, norm_scheme)
        delta_rho, delta_n = _spearman_rho_for_features(delta_features, delta_inference, norm_scheme=norm_scheme)
        rows.append({
            "benchmark": "Raw embeddings - WT",
            "embedding_type": embedding_type,
            "spearman_rho": delta_rho,
            "n_score_sequences": delta_n,
        })
    return pd.DataFrame(rows)


def _exact_popdms_sequence_states():
    if runner.sequence_dataframe is None or runner.sequence_to_protein_sequence is None:
        runner.process_raw_data(drop_stop_codons=True)
    sequence_map = runner.sequence_to_protein_sequence
    mutation_site_map = runner.sequence_to_mutation_sites or {}
    wildtype_key = runner.input_data.wildtype_key
    if wildtype_key not in sequence_map:
        raise KeyError(f"Wildtype key {wildtype_key!r} is missing from sequence_to_protein_sequence.")
    wt_sequence = sequence_map[wildtype_key]
    seq_ids = sorted(runner.sequence_dataframe["SequenceIndex"].astype(str).unique())

    seq_to_states = {}
    observed_sites = set()
    for seq_id in seq_ids:
        protein_sequence = sequence_map.get(seq_id)
        if protein_sequence is None:
            continue
        states = []
        for site_idx in mutation_site_map.get(seq_id, []):
            alt_aa = protein_sequence[site_idx]
            wt_aa = wt_sequence[site_idx]
            if alt_aa != wt_aa:
                site = site_idx + 1
                states.append((site, alt_aa))
                observed_sites.add(site)
        seq_to_states[seq_id] = sorted(states, key=lambda item: item[0])

    sites = sorted(observed_sites)
    ref_aas_by_site = {site: wt_sequence[site - 1] for site in sites}
    return seq_to_states, sites, ref_aas_by_site


def _exact_popdms_frequency_paths(replicates):
    paths = []
    for rep in replicates:
        paths.extend([
            Path(paperPop.get_reads_file(REGULAR_POPDMS_DIR, EXACT_POPDMS_NAME, rep, file_ext=".csv")),
            Path(paperPop.get_aa_freq_file(REGULAR_POPDMS_DIR, EXACT_POPDMS_NAME, "single", rep)),
            Path(paperPop.get_aa_freq_file(REGULAR_POPDMS_DIR, EXACT_POPDMS_NAME, "double", rep)),
        ])
    return paths


def _write_exact_popdms_frequency_files(force_recompute=False):
    REGULAR_POPDMS_DIR.mkdir(parents=True, exist_ok=True)
    if runner.sequence_dataframe is None or runner.sequence_to_protein_sequence is None:
        runner.process_raw_data(drop_stop_codons=True)
    sequence_df = runner.sequence_dataframe.copy()
    sequence_df["SequenceIndex"] = sequence_df["SequenceIndex"].astype(str)
    sequence_df["Frequency"] = pd.to_numeric(sequence_df["Frequency"], errors="coerce").fillna(0.0)
    replicates = sorted(sequence_df["Replicate"].dropna().astype(int).unique())
    existing_paths = _exact_popdms_frequency_paths(replicates)
    if existing_paths and all(path.is_file() for path in existing_paths) and not force_recompute:
        return replicates

    seq_to_states, sites, ref_aas_by_site = _exact_popdms_sequence_states()
    if not sites:
        raise ValueError("No amino-acid substitution sites were available for exact popDMS.")
    q_states = list(paperPop.AA)

    for rep in replicates:
        rep_df = sequence_df[sequence_df["Replicate"].astype(int).eq(rep)].copy()
        generations = sorted(rep_df["Generation"].dropna().unique())
        reads_rows = []
        single_rows = []
        double_rows = []

        for generation in generations:
            time_df = rep_df[rep_df["Generation"].eq(generation)]
            total_count = float(time_df["Frequency"].sum())
            reads_rows.append({"generation": generation, "reads": total_count})
            if total_count <= 0:
                continue

            single_counts = {}
            for site in sites:
                ref_aa = ref_aas_by_site[site]
                for aa in q_states:
                    single_counts[(site, aa)] = 0.0
                single_counts[(site, ref_aa)] = total_count

            double_counts = {}
            for site_i, site_j in itertools.combinations(sites, 2):
                double_counts[(site_i, ref_aas_by_site[site_i], site_j, ref_aas_by_site[site_j])] = total_count

            for _, row in time_df.iterrows():
                count = float(row["Frequency"])
                if count <= 0:
                    continue
                states = seq_to_states.get(row["SequenceIndex"], [])
                if not states:
                    continue
                for site, aa in states:
                    single_counts[(site, aa)] += count
                for (site_i, aa_i), (site_j, aa_j) in itertools.combinations(states, 2):
                    key = (site_i, aa_i, site_j, aa_j)
                    double_counts[key] = double_counts.get(key, 0.0) + count

            for seq_i, site_i in enumerate(sites):
                ref_i = ref_aas_by_site[site_i]
                for aa_i in q_states:
                    if aa_i == ref_i:
                        continue
                    total_single = single_counts[(site_i, aa_i)]
                    for site_j in sites[:seq_i]:
                        total_double = 0.0
                        ref_j = ref_aas_by_site[site_j]
                        for aa_j in q_states:
                            if aa_j != ref_j:
                                total_double += double_counts.get((site_j, aa_j, site_i, aa_i), 0.0)
                        double_counts[(site_j, ref_j, site_i, aa_i)] = total_single - total_double
                    for site_j in sites[seq_i + 1:]:
                        total_double = 0.0
                        ref_j = ref_aas_by_site[site_j]
                        for aa_j in q_states:
                            if aa_j != ref_j:
                                total_double += double_counts.get((site_i, aa_i, site_j, aa_j), 0.0)
                        double_counts[(site_i, aa_i, site_j, ref_j)] = total_single - total_double

            for (site, aa), count in list(single_counts.items()):
                if aa != ref_aas_by_site[site]:
                    single_counts[(site, ref_aas_by_site[site])] -= count

            for (site_i, aa_i, site_j, aa_j), count in list(double_counts.items()):
                if aa_i != ref_aas_by_site[site_i] or aa_j != ref_aas_by_site[site_j]:
                    wt_key = (site_i, ref_aas_by_site[site_i], site_j, ref_aas_by_site[site_j])
                    double_counts[wt_key] = double_counts.get(wt_key, 0.0) - count

            for (site, aa), count in single_counts.items():
                if count > 0:
                    single_rows.append({
                        "generation": generation,
                        "site": site,
                        "aa": aa,
                        "frequency": count / total_count,
                        "WT_indicator": aa == ref_aas_by_site[site],
                    })

            for (site_i, aa_i, site_j, aa_j), count in double_counts.items():
                if count > 0:
                    double_rows.append({
                        "generation": generation,
                        "site_1": site_i,
                        "aa_1": aa_i,
                        "site_2": site_j,
                        "aa_2": aa_j,
                        "frequency": count / total_count,
                    })

        pd.DataFrame(reads_rows).to_csv(
            paperPop.get_reads_file(REGULAR_POPDMS_DIR, EXACT_POPDMS_NAME, rep, file_ext=".csv"),
            index=False,
        )
        pd.DataFrame(single_rows).to_csv(
            paperPop.get_aa_freq_file(REGULAR_POPDMS_DIR, EXACT_POPDMS_NAME, "single", rep),
            index=False,
            compression="gzip",
        )
        pd.DataFrame(double_rows).to_csv(
            paperPop.get_aa_freq_file(REGULAR_POPDMS_DIR, EXACT_POPDMS_NAME, "double", rep),
            index=False,
            compression="gzip",
        )
    return replicates


def _exact_popdms_selection_coefficients(force_recompute=False, allow_compute=False):
    needs_compute = force_recompute or not EXACT_POPDMS_SELECTION_PATH.is_file()
    if needs_compute and not allow_compute:
        raise FileNotFoundError(
            f"Cached exact popDMS selection coefficients are missing at {EXACT_POPDMS_SELECTION_PATH}. "
            "Run the exact popDMS inference cell with RUN_EXACT_POPDMS_INFERENCE=True first."
        )
    if force_recompute and EXACT_POPDMS_SELECTION_PATH.is_file():
        EXACT_POPDMS_SELECTION_PATH.unlink()
    if not EXACT_POPDMS_SELECTION_PATH.is_file():
        replicates = _write_exact_popdms_frequency_files(force_recompute=force_recompute)
        paperPop.infer_correlated(
            EXACT_POPDMS_NAME,
            n_replicates=len(replicates),
            corr_cutoff_pct=EXACT_POPDMS_CORR_CUTOFF_PCT,
            gamma=EXACT_POPDMS_GAMMA,
            norm_WT=EXACT_POPDMS_NORM_WT,
            freq_dir=str(REGULAR_POPDMS_DIR),
            output_dir=str(REGULAR_POPDMS_DIR),
            with_epistasis=False,
            plot_gamma=False,
        )
    selection_df = pd.read_csv(EXACT_POPDMS_SELECTION_PATH, compression="gzip")
    selection_df.to_csv(EXACT_POPDMS_SELECTION_TABLE_PATH, index=False)
    return selection_df


def _exact_popdms_fitness_dataframe(selection_df, annotation_map=None):
    seq_to_states, _, _ = _exact_popdms_sequence_states()
    non_wt_selection = selection_df[~selection_df["WT_indicator"].astype(bool)].copy()
    non_wt_selection["site"] = non_wt_selection["site"].astype(int)
    selection_lookup = {
        (int(row["site"]), str(row["amino_acid"])): float(row["joint"])
        for _, row in non_wt_selection.iterrows()
    }
    seq_ids = sorted(runner.sequence_dataframe["SequenceIndex"].astype(str).unique())
    rows = []
    for seq_id in seq_ids:
        s_sum = sum(selection_lookup.get((site, aa), 0.0) for site, aa in seq_to_states.get(seq_id, []))
        rows.append({"SequenceIndex": seq_id, "fitness": 1.0 + s_sum})
    fitness_df = pd.DataFrame(rows)
    if annotation_map is not None:
        fitness_df["annotation"] = fitness_df["SequenceIndex"].map(annotation_map)
    return fitness_df


def _spearman_for_fitness_dataframe(fitness_df, score_col="score"):
    if runner.scores_dataframe is None:
        runner.load_functional_scores()
    score_df = runner.scores_dataframe[["SequenceIndex", score_col]].dropna().copy()
    comparison_df = fitness_df.merge(score_df, on="SequenceIndex", how="inner")
    finite = np.isfinite(comparison_df["fitness"]) & np.isfinite(comparison_df[score_col])
    if finite.sum() < 3:
        return np.nan, int(finite.sum())
    rho = spearmanr(comparison_df.loc[finite, "fitness"], comparison_df.loc[finite, score_col]).statistic
    return float(rho), int(finite.sum())


def _exact_popdms_spearman_benchmark_row():
    selection_df = _exact_popdms_selection_coefficients(
        force_recompute=FORCE_RECOMPUTE_EXACT_POPDMS_BASELINE,
    )
    fitness_df = _exact_popdms_fitness_dataframe(selection_df)
    fitness_df.to_csv(EXACT_POPDMS_FITNESS_PATH, index=False)
    spearman_rho, n_score_sequences = _spearman_for_fitness_dataframe(fitness_df)
    return {
        "benchmark": "Regular popDMS",
        "embedding_type": "regular popDMS",
        "spearman_rho": spearman_rho,
        "n_score_sequences": n_score_sequences,
        "model_label": "Regular popDMS",
        "method": "regular popDMS",
        "model_group": "Regular popDMS",
        "n_features": int(len(selection_df)),
        "feature_path": "",
        "inference_path": str(EXACT_POPDMS_SELECTION_PATH),
        "method_pool": "regular popDMS",
    }


def _pareto_mask_maximize(df, x_col, y_col):
    values = df[[x_col, y_col]].to_numpy(dtype=float)
    finite = np.isfinite(values).all(axis=1)
    mask = np.zeros(len(df), dtype=bool)
    finite_indices = np.flatnonzero(finite)
    finite_values = values[finite]
    for local_i, point in enumerate(finite_values):
        dominates = (
            (finite_values[:, 0] >= point[0])
            & (finite_values[:, 1] >= point[1])
            & ((finite_values[:, 0] > point[0]) | (finite_values[:, 1] > point[1]))
        )
        if not dominates.any():
            mask[finite_indices[local_i]] = True
    return mask


def _ensure_sweep_breakdown_columns(df):
    df = df.copy()

    def _embedding_type_from_label(run_label):
        run_label = str(run_label)
        if run_label.startswith("mean_pool") or "_mean_pool_" in run_label:
            return "mean_pool"
        if run_label.startswith("max_pool") or "_max_pool_" in run_label:
            return "max_pool"
        return "unknown"

    def _method_from_label(run_label):
        run_label = str(run_label)
        if run_label.startswith("DeltaEmbSAE") or "DeltaEmbSAE" in run_label:
            return "DeltaEmbSAE"
        if run_label.startswith("DeltaSAE") or "DeltaSAE" in run_label:
            return "DeltaSAE"
        if run_label.startswith("SAE") or "_SAE_" in run_label:
            return "SAE"
        return "unknown"

    label_embedding_type = df.get("run_label", pd.Series(index=df.index, dtype=object)).map(_embedding_type_from_label)
    if "embedding_type" in df.columns:
        existing_embedding_type = df["embedding_type"].fillna("unknown").astype(str)
        df["embedding_type"] = existing_embedding_type.where(label_embedding_type == "unknown", label_embedding_type)
    else:
        df["embedding_type"] = label_embedding_type

    label_method = df.get("run_label", pd.Series(index=df.index, dtype=object)).map(_method_from_label)
    if "method" in df.columns:
        existing_method = df["method"].fillna("unknown").astype(str)
        df["method"] = existing_method.where(label_method == "unknown", label_method)
    else:
        df["method"] = label_method
    df["method_pool"] = df["method"].astype(str) + " / " + df["embedding_type"].astype(str)
    return df


def _ensure_embedding_type_column(df):
    return _ensure_sweep_breakdown_columns(df)


def _plot_pareto(
    df,
    x_col,
    y_col,
    x_label,
    y_label,
    output_path,
    title_prefix="DeltaSAE sweep",
    group_front_col="method_pool",
    style_col="method",
    size_col="k",
    benchmark_df=None,
):
    plot_df = df[np.isfinite(df[x_col]) & np.isfinite(df[y_col])].copy()
    if plot_df.empty:
        print(f"No finite values available for {x_label} vs {y_label}.")
        return None
    plot_df = _ensure_sweep_breakdown_columns(plot_df)
    plot_df["pareto"] = _pareto_mask_maximize(plot_df, x_col, y_col)
    pareto_df = plot_df[plot_df["pareto"]].sort_values(x_col)
    method_order = [method for method in ["SAE", "DeltaSAE", "DeltaEmbSAE"] if method in set(plot_df["method"])]
    method_order.extend(sorted(set(plot_df["method"]) - set(method_order)))
    pool_order = [pool for pool in ["mean_pool", "max_pool"] if pool in set(plot_df["embedding_type"])]
    pool_order.extend(sorted(set(plot_df["embedding_type"]) - set(pool_order)))
    pool_line_palette = {"mean_pool": "#d55e00", "max_pool": "#0072b2", "unknown": "#666666"}
    method_linestyle = {"SAE": "-.", "DeltaSAE": "-", "DeltaEmbSAE": "--", "unknown": ":"}
    benchmark_linestyle = {"Raw embeddings": ":", "Raw embeddings - WT": "-.", "Regular popDMS": "--"}

    fig, ax, legend_anchor = _square_axes_figure()
    if len(pareto_df):
        ax.plot(
            pareto_df[x_col],
            pareto_df[y_col],
            color="black",
            linewidth=2.3,
            alpha=0.6,
            zorder=1,
            label="Overall Pareto front",
        )
    if group_front_col is not None and group_front_col in plot_df.columns and plot_df[group_front_col].nunique() > 1:
        for group_value, group_df in plot_df.groupby(group_front_col, sort=True):
            group_df = group_df.copy()
            group_df["group_pareto"] = _pareto_mask_maximize(group_df, x_col, y_col)
            group_pareto_df = group_df[group_df["group_pareto"]].sort_values(x_col)
            if len(group_pareto_df):
                pool = str(group_pareto_df["embedding_type"].iloc[0])
                method = str(group_pareto_df["method"].iloc[0])
                ax.plot(
                    group_pareto_df[x_col],
                    group_pareto_df[y_col],
                    color=pool_line_palette.get(pool, "#666666"),
                    linewidth=1.8,
                    linestyle=method_linestyle.get(method, ":"),
                    alpha=0.8,
                    zorder=1,
                    label=f"{group_value} Pareto front",
                )
    scatter_kwargs = {
        "data": plot_df,
        "x": x_col,
        "y": y_col,
        "hue": "n_features",
        "palette": "viridis",
        "alpha": 0.88,
        "edgecolor": "white",
        "linewidth": 0.6,
        "zorder": 3,
        "ax": ax,
    }
    if style_col is not None and style_col in plot_df.columns and plot_df[style_col].nunique() > 1:
        scatter_kwargs["style"] = style_col
    elif "k" in plot_df.columns and plot_df["k"].nunique() > 1:
        scatter_kwargs["style"] = "k"
    if size_col is None:
        scatter_kwargs["s"] = 70
    elif size_col in plot_df.columns and plot_df[size_col].nunique() > 1:
        scatter_kwargs["size"] = size_col
        scatter_kwargs["sizes"] = (45, 130)
    else:
        scatter_kwargs["s"] = 70
    sns.scatterplot(**scatter_kwargs)
    if benchmark_df is not None and y_col == "spearman_rho":
        visible_pools = set(plot_df["embedding_type"])
        visible_benchmarks = benchmark_df[
            benchmark_df["embedding_type"].isin(visible_pools)
            | benchmark_df["benchmark"].eq("Regular popDMS")
        ]
        for _, benchmark_row in visible_benchmarks.iterrows():
            benchmark_rho = benchmark_row.get("spearman_rho", np.nan)
            if not np.isfinite(benchmark_rho):
                continue
            benchmark_name = benchmark_row["benchmark"]
            benchmark_pool = benchmark_row["embedding_type"]
            is_popdms = benchmark_name == "Regular popDMS"
            ax.axhline(
                benchmark_rho,
                color="#111111" if is_popdms else pool_line_palette.get(benchmark_pool, "#666666"),
                linestyle=benchmark_linestyle.get(benchmark_name, ":"),
                linewidth=2.0 if is_popdms else 1.6,
                alpha=0.95 if is_popdms else 0.9,
                zorder=0,
                label=(
                    f"Regular popDMS rho={benchmark_rho:.3f}"
                    if is_popdms else f"{benchmark_pool} {benchmark_name} rho={benchmark_rho:.3f}"
                ),
            )
    if x_col == "reconstruction_r2":
        ax.set_xlim(0, 1 + R2_AXIS_EPSILON)
    if y_col == "reconstruction_r2":
        ax.set_ylim(0, 1 + R2_AXIS_EPSILON)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(_latex_safe_label(f"{title_prefix}: {x_label} vs {y_label}"))
    _external_legend(fig, ax, legend_anchor)
    fig.savefig(output_path, dpi=FIGURE_DPI)
    return fig


def _plot_suffix(value):
    return str(value).replace(" / ", "_").replace(" ", "_").replace("/", "_")




In [ ]:
RUN_EXACT_POPDMS_INFERENCE = False

if RUN_EXACT_POPDMS_INFERENCE:
    selection_df = _exact_popdms_selection_coefficients(
        force_recompute=FORCE_RECOMPUTE_EXACT_POPDMS_BASELINE,
        allow_compute=True,
    )
    fitness_df = _exact_popdms_fitness_dataframe(selection_df)
    selection_df.to_csv(EXACT_POPDMS_SELECTION_TABLE_PATH, index=False)
    fitness_df.to_csv(EXACT_POPDMS_FITNESS_PATH, index=False)
    print(f"Saved exact popDMS selection coefficients to {EXACT_POPDMS_SELECTION_PATH}")
    display(selection_df.head())
    display(fitness_df.head())
elif EXACT_POPDMS_SELECTION_PATH.is_file():
    print(f"Using cached exact popDMS selection coefficients: {EXACT_POPDMS_SELECTION_PATH}")
else:
    print(
        "Exact popDMS cache is missing. Set RUN_EXACT_POPDMS_INFERENCE=True "
        "in this cell to run paperPop.infer_correlated before plotting popDMS baselines."
    )


In [ ]:
if RUN_SAE_SWEEP_PARETO_ANALYSIS:
    existing_sweep_result_paths = [path for path in SAE_SWEEP_RESULTS_PATHS if path.is_file()]
    if not existing_sweep_result_paths:
        print(f"No completed sweep summaries found under {SAE_SWEEP_ANALYSIS_DIR}.")
        print("Submit or run the SAE sweep jobs above, then rerun this cell after sae_sweep_results.csv files are written.")
    else:
        print(f"Loaded {len(existing_sweep_result_paths)} sweep result CSV(s) from {SAE_SWEEP_ANALYSIS_DIR}.")
        sweep_result_frames = []
        for path in existing_sweep_result_paths:
            frame = pd.read_csv(path)
            frame["sweep_results_path"] = str(path)
            sweep_result_frames.append(frame)
        sweep_results = pd.concat(sweep_result_frames, ignore_index=True)
        metric_rows = []
        for _, row in sweep_results.iterrows():
            row_dict = row.to_dict()
            metric_row = dict(row_dict)
            metric_row.update(_reconstruction_metrics(row_dict.get("viz_path", "")))
            rho, n_score = _spearman_rho_for_run(
                row_dict.get("feature_path", ""),
                row_dict.get("inference_path", ""),
                norm_scheme=NORM_SCHEME,
                score_col="score",
            )
            metric_row["spearman_rho"] = rho
            metric_row["n_score_sequences"] = n_score
            metric_rows.append(metric_row)

        sae_sweep_metrics = _ensure_sweep_breakdown_columns(pd.DataFrame(metric_rows))
        sae_sweep_metrics.to_csv(TABLE_DIR / "BRCA1_DeltaSAE_sweep_metrics.csv", index=False)
        benchmark_embedding_types = [
            embedding_type
            for embedding_type in ["mean_pool", "max_pool"]
            if embedding_type in set(sae_sweep_metrics["embedding_type"])
        ]
        raw_embedding_benchmarks = (
            _raw_embedding_benchmark_rows(benchmark_embedding_types, REPRESENTATIVE_LAYER, NORM_SCHEME)
            if RUN_RAW_EMBEDDING_BENCHMARKS and benchmark_embedding_types
            else pd.DataFrame(columns=["benchmark", "embedding_type", "spearman_rho", "n_score_sequences"])
        )
        if RUN_EXACT_POPDMS_BASELINE:
            try:
                raw_embedding_benchmarks = pd.concat(
                    [raw_embedding_benchmarks, pd.DataFrame([_exact_popdms_spearman_benchmark_row()])],
                    ignore_index=True,
                    sort=False,
                )
            except Exception as exc:
                print(f"Skipping regular popDMS Spearman baseline: {exc}")
        raw_embedding_benchmarks.to_csv(TABLE_DIR / "BRCA1_raw_embedding_spearman_benchmarks.csv", index=False)
        display(raw_embedding_benchmarks)
        display(
            sae_sweep_metrics.sort_values("spearman_rho", ascending=False)[[
                "run_label",
                "method",
                "embedding_type",
                "n_features",
                "k",
                "activation_sparsity",
                "reconstruction_r2",
                "spearman_rho",
                "n_active_features",
                "n_score_sequences",
                "method_pool",
            ]].head(20)
        )

        pareto_specs = [
            (
                "activation_sparsity",
                "reconstruction_r2",
                "Activation sparsity",
                "Reconstruction R2",
                FIGURE_DIR / "BRCA1_DeltaSAE_sweep_pareto_sparsity_vs_reconstruction.png",
            ),
            (
                "activation_sparsity",
                "spearman_rho",
                "Activation sparsity",
                "MaveDB Spearman rho",
                FIGURE_DIR / "BRCA1_DeltaSAE_sweep_pareto_sparsity_vs_spearman.png",
            ),
            (
                "reconstruction_r2",
                "spearman_rho",
                "Reconstruction R2",
                "MaveDB Spearman rho",
                FIGURE_DIR / "BRCA1_DeltaSAE_sweep_pareto_reconstruction_vs_spearman.png",
            ),
        ]
        for x_col, y_col, x_label, y_label, output_path in pareto_specs:
            fig = _plot_pareto(
                sae_sweep_metrics,
                x_col,
                y_col,
                x_label,
                y_label,
                output_path,
                title_prefix="All SAE sweeps",
                group_front_col="method_pool",
                style_col="method",
                size_col="k",
                benchmark_df=raw_embedding_benchmarks,
            )
            if fig is not None:
                plt.show()

            for method, method_df in sae_sweep_metrics.groupby("method", sort=True):
                if method == "unknown":
                    continue
                method_output_path = output_path.with_name(f"{output_path.stem}_method_{_plot_suffix(method)}{output_path.suffix}")
                method_fig = _plot_pareto(
                    method_df,
                    x_col,
                    y_col,
                    x_label,
                    y_label,
                    method_output_path,
                    title_prefix=f"{method} sweeps",
                    group_front_col="embedding_type",
                    style_col="embedding_type",
                    size_col="k",
                    benchmark_df=raw_embedding_benchmarks,
                )
                if method_fig is not None:
                    plt.show()

            if PLOT_POOL_ONLY_PARETO:
                for embedding_type, pool_df in sae_sweep_metrics.groupby("embedding_type", sort=True):
                    if embedding_type == "unknown":
                        continue
                    pool_output_path = output_path.with_name(f"{output_path.stem}_pool_{_plot_suffix(embedding_type)}{output_path.suffix}")
                    pool_fig = _plot_pareto(
                        pool_df,
                        x_col,
                        y_col,
                        x_label,
                        y_label,
                        pool_output_path,
                        title_prefix=f"{embedding_type} sweeps",
                        group_front_col="method",
                        style_col="method",
                        size_col="k",
                        benchmark_df=raw_embedding_benchmarks,
                    )
                    if pool_fig is not None:
                        plt.show()

            if PLOT_METHOD_POOL_PARETO:
                for method_pool, group_df in sae_sweep_metrics.groupby("method_pool", sort=True):
                    if "unknown" in method_pool:
                        continue
                    group_output_path = output_path.with_name(f"{output_path.stem}_group_{_plot_suffix(method_pool)}{output_path.suffix}")
                    group_fig = _plot_pareto(
                        group_df,
                        x_col,
                        y_col,
                        x_label,
                        y_label,
                        group_output_path,
                        title_prefix=f"{method_pool} sweeps",
                        group_front_col=None,
                        style_col="k",
                        size_col=None,
                        benchmark_df=raw_embedding_benchmarks,
                    )
                    if group_fig is not None:
                        plt.show()


        def _gamma_regularization_metrics_for_features(
            seq_to_features,
            model_metadata,
            norm_scheme=NORM_SCHEME,
            gamma_values=GAMMA_REGULARIZATION_VALUES,
        ):
            sequence_dataframe, seq_to_features = runner._drop_missing_features(
                runner.sequence_dataframe,
                seq_to_features,
                f"Gamma regularization sweep for {model_metadata.get('model_label', 'model')}",
            )
            runner._require_vector_features(seq_to_features, "Gamma regularization sweep")
            if norm_scheme is not None and norm_scheme != "none":
                seq_ids = list(seq_to_features)
                features = np.asarray([seq_to_features[seq_id] for seq_id in seq_ids], dtype=float)
                features = esmDMS._normalize_features(features, norm_scheme)
                seq_to_features = dict(zip(seq_ids, features))

            gamma_values_out, s_by_gamma, _ = infer_gamma_range(
                sequence_dataframe,
                seq_to_features,
                gamma_values=gamma_values,
            )
            rows = []
            for gamma_idx, gamma in enumerate(gamma_values_out):
                pair_corrs = []
                for rep_i, rep_j in runner._rep_pairs(s_by_gamma.shape[1]):
                    rho = runner._safe_corr(s_by_gamma[gamma_idx, rep_i], s_by_gamma[gamma_idx, rep_j], pearsonr)
                    pair_corrs.append(rho)
                    rows.append({
                        **model_metadata,
                        "gamma": gamma,
                        "rep_i": rep_i + 1,
                        "rep_j": rep_j + 1,
                        "pearson_r": rho,
                    })
                rows.append({
                    **model_metadata,
                    "gamma": gamma,
                    "rep_i": "mean",
                    "rep_j": "mean",
                    "pearson_r": np.nanmean(pair_corrs) if pair_corrs else np.nan,
                })
            return pd.DataFrame(rows)


        def _gamma_model_specs(sweep_metrics, benchmark_embedding_types):
            specs = []
            completed = sweep_metrics.copy()
            completed = completed[completed["feature_path"].map(lambda path: _as_existing_path(path) is not None)].copy()
            for _, row in completed.iterrows():
                row_dict = row.to_dict()
                method = row_dict.get("method", "unknown")
                embedding_type = row_dict.get("embedding_type", "unknown")
                run_label = row_dict.get("run_label", "model")
                specs.append({
                    "model_label": f"{method} {embedding_type} {run_label}",
                    "method": method,
                    "embedding_type": embedding_type,
                    "run_label": run_label,
                    "n_features": row_dict.get("n_features", np.nan),
                    "k": row_dict.get("k", np.nan),
                    "seed": row_dict.get("seed", np.nan),
                    "feature_path": row_dict.get("feature_path", ""),
                    "baseline_kind": "sweep",
                    "is_raw_esm_baseline": False,
                })

            baseline_embedding_types = set(benchmark_embedding_types)
            baseline_embedding_types.update(globals().get("SAE_SWEEP_EMBEDDING_TYPES", []))
            baseline_embedding_types.add(SAE_EMBEDDING_TYPE)
            for embedding_type in sorted(baseline_embedding_types):
                raw_path = runner._embedding_path(REPRESENTATIVE_LAYER, embedding_type)
                if raw_path.is_file():
                    specs.append({
                        "model_label": f"raw ESM {embedding_type}",
                        "method": "none",
                        "embedding_type": embedding_type,
                        "run_label": f"raw_esm_{embedding_type}",
                        "n_features": np.nan,
                        "k": np.nan,
                        "seed": np.nan,
                        "feature_path": str(raw_path),
                        "baseline_kind": "raw",
                        "is_raw_esm_baseline": True,
                    })
                try:
                    delta_features = _raw_delta_embedding_features(REPRESENTATIVE_LAYER, embedding_type)
                except Exception as exc:
                    print(f"Skipping raw ESM - WT gamma sweep for {embedding_type}: {exc}")
                else:
                    specs.append({
                        "model_label": f"raw ESM - WT {embedding_type}",
                        "method": "none_delta_wt",
                        "embedding_type": embedding_type,
                        "run_label": f"raw_esm_delta_wt_{embedding_type}",
                        "n_features": np.nan,
                        "k": np.nan,
                        "seed": np.nan,
                        "feature_path": "",
                        "baseline_kind": "raw_delta_wt",
                        "is_raw_esm_baseline": True,
                    })
            return specs


        def _plot_gamma_regularization_curves(gamma_df, output_path):
            mean_df = gamma_df[gamma_df["rep_i"] == "mean"].copy()
            if mean_df.empty:
                print("No gamma regularization values available to plot.")
                return None
            fig, ax, legend_anchor = _square_axes_figure()
            palette = dict(zip(
                sorted(mean_df["embedding_type"].dropna().unique()),
                sns.color_palette("Set2", n_colors=mean_df["embedding_type"].nunique()),
            ))
            for _, model_df in mean_df.groupby("model_label", sort=False):
                row0 = model_df.iloc[0]
                color = palette.get(row0["embedding_type"], "0.5")
                if bool(row0["is_raw_esm_baseline"]):
                    ax.plot(
                        model_df["gamma"],
                        model_df["pearson_r"],
                        color=color,
                        linewidth=3.0,
                        marker="o",
                        label=row0["model_label"],
                    )
                else:
                    ax.plot(
                        model_df["gamma"],
                        model_df["pearson_r"],
                        color=color,
                        alpha=0.28,
                        linewidth=1.2,
                    )
            ax.set_xscale("log")
            ax.set_ylim(-1, 1)
            ax.set_xlabel("Regularization strength (gamma)")
            ax.set_ylabel("Mean replicate selection-coefficient Pearson r")
            ax.set_title("Gamma regularization across inferred models")
            handles, labels = ax.get_legend_handles_labels()
            if handles:
                _external_legend(fig, ax, legend_anchor, handles, labels)
            fig.savefig(output_path, dpi=FIGURE_DPI)
            return fig


        def _plot_gamma_optimum_summary(gamma_df, output_path):
            mean_df = gamma_df[gamma_df["rep_i"] == "mean"].copy()
            if mean_df.empty:
                return None
            best_idx = mean_df.groupby("model_label")["pearson_r"].idxmax()
            best_df = mean_df.loc[best_idx].copy().sort_values(["method", "embedding_type", "gamma"])
            best_df["model_family"] = np.where(
                best_df["is_raw_esm_baseline"],
                "raw ESM",
                best_df["method"].astype(str) + " k=" + best_df["k"].astype(str) + " nf=" + best_df["n_features"].astype(str),
            )
            ax_size = max(PLOT_AX_SIZE, min(9.0, 0.22 * len(best_df)))
            fig, ax, legend_anchor = _square_axes_figure(ax_size=ax_size)
            sns.scatterplot(
                data=best_df,
                x="gamma",
                y="model_family",
                hue="embedding_type",
                size="pearson_r",
                sizes=(35, 180),
                alpha=0.75,
                ax=ax,
            )
            ax.set_xscale("log")
            ax.set_xlabel("Gamma with highest mean replicate Pearson r")
            ax.set_ylabel("")
            ax.set_title("Best gamma by inferred model")
            _external_legend(fig, ax, legend_anchor)
            fig.savefig(output_path, dpi=FIGURE_DPI)
            return fig


        if PLOT_GAMMA_REGULARIZATION_FOR_ALL_MODELS:
            gamma_rows = []
            for model_spec in _gamma_model_specs(sae_sweep_metrics, benchmark_embedding_types):
                baseline_kind = model_spec.pop("baseline_kind", "sweep")
                if baseline_kind == "raw":
                    seq_to_features = runner.load_embeddings(REPRESENTATIVE_LAYER, model_spec["embedding_type"])
                elif baseline_kind == "raw_delta_wt":
                    seq_to_features = _raw_delta_embedding_features(REPRESENTATIVE_LAYER, model_spec["embedding_type"])
                else:
                    seq_to_features = _load_pickle(model_spec["feature_path"])
                reg_df = _gamma_regularization_metrics_for_features(
                    seq_to_features,
                    model_spec,
                    norm_scheme=NORM_SCHEME,
                    gamma_values=GAMMA_REGULARIZATION_VALUES,
                )
                if not reg_df.empty:
                    gamma_rows.append(reg_df)
            if gamma_rows:
                gamma_regularization_df = pd.concat(gamma_rows, ignore_index=True)
                gamma_regularization_df.to_csv(
                    TABLE_DIR / "BRCA1_all_models_gamma_regularization.csv",
                    index=False,
                )
                display(
                    gamma_regularization_df[gamma_regularization_df["rep_i"] == "mean"]
                    .sort_values("pearson_r", ascending=False)[[
                        "model_label",
                        "method",
                        "embedding_type",
                        "n_features",
                        "k",
                        "gamma",
                        "pearson_r",
                        "is_raw_esm_baseline",
                    ]]
                    .head(20)
                )
                fig = _plot_gamma_regularization_curves(
                    gamma_regularization_df,
                    FIGURE_DIR / "BRCA1_all_models_gamma_regularization_curves.png",
                )
                if fig is not None:
                    plt.show()
                fig = _plot_gamma_optimum_summary(
                    gamma_regularization_df,
                    FIGURE_DIR / "BRCA1_all_models_gamma_regularization_optimum_gamma.png",
                )
                if fig is not None:
                    plt.show()
            else:
                print("No feature files were available for gamma regularization plots.")


## Benign Vs Pathogenic Classification

Use the BRCA1 pathogenicity annotations to evaluate whether inferred fitness separates benign from pathogenic variants. Pathogenic is treated as the positive class and `-fitness` is used as the classifier score.


In [ ]:
RUN_PATHOGENICITY_CLASSIFICATION = True
PATHOGENICITY_LABELS = {"benign", "pathogenic"}


def _annotation_map_for_binary_pathogenicity():
    if not hasattr(runner, "primary_key_to_annotation"):
        annotation_df = _load_brca1_pathogenicity_annotations(
            ANNOTATION_PATH,
            BRCA1_INPUT.scores_csv_path,
            BRCA1_INPUT.primary_key,
        )
        runner.annotation_dataframe = annotation_df
        runner.primary_key_to_annotation_identifier = dict(zip(
            annotation_df[BRCA1_INPUT.primary_key],
            annotation_df["annotation_identifier"],
        ))
        runner.primary_key_to_annotation = dict(zip(
            annotation_df[BRCA1_INPUT.primary_key],
            annotation_df["annotation"],
        ))
    return {
        seq_id: annotation
        for seq_id, annotation in runner.primary_key_to_annotation.items()
        if annotation in PATHOGENICITY_LABELS
    }


def _rank_auc(labels, scores):
    labels = np.asarray(labels, dtype=bool)
    scores = np.asarray(scores, dtype=float)
    finite = np.isfinite(scores)
    labels = labels[finite]
    scores = scores[finite]
    n_pos = int(labels.sum())
    n_neg = int((~labels).sum())
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = pd.Series(scores).rank(method="average").to_numpy()
    return float((ranks[labels].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))


def _fitness_for_feature_mapping(seq_to_features, inference_result, annotation_map, norm_scheme=NORM_SCHEME):
    seq_ids = [
        seq_id
        for seq_id, feature in seq_to_features.items()
        if seq_id in annotation_map and feature is not None
    ]
    if not seq_ids:
        return pd.DataFrame(columns=["SequenceIndex", "annotation", "fitness"])
    features = np.asarray([seq_to_features[seq_id] for seq_id in seq_ids], dtype=float)
    if norm_scheme is not None and norm_scheme != "none":
        features = esmDMS._normalize_features(features, norm_scheme)
    fitness = 1.0 + features @ inference_result.s_joint
    return pd.DataFrame({
        "SequenceIndex": seq_ids,
        "annotation": [annotation_map[seq_id] for seq_id in seq_ids],
        "fitness": fitness,
    })


def _classification_metrics_for_fitness(fitness_df):
    binary_df = fitness_df[fitness_df["annotation"].isin(PATHOGENICITY_LABELS)].dropna(subset=["fitness"]).copy()
    if binary_df.empty:
        return {"auc": np.nan, "n_benign": 0, "n_pathogenic": 0, "n_variants": 0}
    labels = binary_df["annotation"].eq("pathogenic").to_numpy()
    auc = _rank_auc(labels, -binary_df["fitness"].to_numpy(dtype=float))
    return {
        "auc": auc,
        "n_benign": int((~labels).sum()),
        "n_pathogenic": int(labels.sum()),
        "n_variants": int(len(binary_df)),
    }


def _classification_row(model_spec, annotation_map):
    if "fitness_df" in model_spec:
        fitness_df = model_spec["fitness_df"].copy()
    else:
        seq_to_features = model_spec["seq_to_features"]
        inference_result = model_spec["inference_result"]
        fitness_df = _fitness_for_feature_mapping(seq_to_features, inference_result, annotation_map)
    metrics = _classification_metrics_for_fitness(fitness_df)
    return {
        **{key: value for key, value in model_spec.items() if key not in {"seq_to_features", "inference_result", "fitness_df"}},
        **metrics,
        "fitness_df": fitness_df,
    }


def _sweep_metrics_for_classification():
    if "sae_sweep_metrics" in globals():
        return sae_sweep_metrics.copy()
    metrics_path = TABLE_DIR / "BRCA1_DeltaSAE_sweep_metrics.csv"
    if metrics_path.is_file():
        return pd.read_csv(metrics_path)
    result_paths = [path for path in SAE_SWEEP_RESULTS_PATHS if path.is_file()]
    if not result_paths:
        return pd.DataFrame()
    return pd.concat([pd.read_csv(path) for path in result_paths], ignore_index=True)


def _raw_classification_specs(annotation_map):
    specs = []
    embedding_types = sorted(
        set(globals().get("SAE_SWEEP_EMBEDDING_TYPES", [])) | {SAE_EMBEDDING_TYPE, "mean_pool", "max_pool"}
    )
    for embedding_type in embedding_types:
        try:
            raw_features = runner.load_embeddings(REPRESENTATIVE_LAYER, embedding_type)
            raw_inference = runner.run_feature_inference(
                layer=REPRESENTATIVE_LAYER,
                abstraction_method="none",
                abstraction_params={"norm_scheme": NORM_SCHEME},
                embedding_type=embedding_type,
            )
            specs.append({
                "model_label": f"Raw embeddings {embedding_type}",
                "model_group": "benchmark",
                "benchmark": "Raw embeddings",
                "method": "raw ESM",
                "embedding_type": embedding_type,
                "seq_to_features": raw_features,
                "inference_result": raw_inference,
            })

            delta_features, delta_inference, _, _ = _raw_delta_embedding_inference(
                REPRESENTATIVE_LAYER,
                embedding_type,
                NORM_SCHEME,
            )
            specs.append({
                "model_label": f"Raw embeddings - WT {embedding_type}",
                "model_group": "benchmark",
                "benchmark": "Raw embeddings - WT",
                "method": "raw ESM - WT",
                "embedding_type": embedding_type,
                "seq_to_features": delta_features,
                "inference_result": delta_inference,
            })
        except Exception as exc:
            print(f"Skipping {embedding_type} raw benchmark classification: {exc}")
    return specs


def _exact_popdms_classification_specs(annotation_map):
    if not globals().get("RUN_EXACT_POPDMS_BASELINE", True):
        return []
    try:
        selection_df = _exact_popdms_selection_coefficients(
            force_recompute=FORCE_RECOMPUTE_EXACT_POPDMS_BASELINE,
        )
        fitness_df = _exact_popdms_fitness_dataframe(selection_df, annotation_map)
        fitness_df.to_csv(EXACT_POPDMS_FITNESS_PATH, index=False)
        spearman_rho, n_score_sequences = _spearman_for_fitness_dataframe(fitness_df)
    except Exception as exc:
        print(f"Skipping regular popDMS pathogenicity classification: {exc}")
        return []
    return [{
        "model_label": "Regular popDMS",
        "model_group": "Regular popDMS",
        "benchmark": "Regular popDMS",
        "method": "regular popDMS",
        "embedding_type": "amino-acid haplotype frequencies",
        "n_features": int(len(selection_df)),
        "k": np.nan,
        "spearman_rho": spearman_rho,
        "n_score_sequences": n_score_sequences,
        "feature_path": "",
        "inference_path": str(EXACT_POPDMS_SELECTION_PATH),
        "method_pool": "regular popDMS",
        "fitness_df": fitness_df,
    }]


def _sae_classification_specs(sweep_df):
    specs = []
    if sweep_df.empty:
        return specs
    ok_df = sweep_df[sweep_df.get("status", "ok").eq("ok")].copy() if "status" in sweep_df.columns else sweep_df.copy()
    for _, row in ok_df.iterrows():
        feature_path = _as_existing_path(row.get("feature_path"))
        inference_path = _as_existing_path(row.get("inference_path"))
        if feature_path is None or inference_path is None:
            continue
        specs.append({
            "model_label": row.get("run_label", feature_path.parent.name),
            "model_group": "SAE sweep",
            "benchmark": "SAE sweep",
            "method": row.get("method", "unknown"),
            "embedding_type": row.get("embedding_type", "unknown"),
            "n_features": row.get("n_features", np.nan),
            "k": row.get("k", np.nan),
            "spearman_rho": row.get("spearman_rho", np.nan),
            "feature_path": str(feature_path),
            "inference_path": str(inference_path),
            "seq_to_features": _load_pickle(feature_path),
            "inference_result": _load_pickle(inference_path),
        })
    return specs


if RUN_PATHOGENICITY_CLASSIFICATION:
    annotation_map = _annotation_map_for_binary_pathogenicity()
    sweep_df = _ensure_sweep_breakdown_columns(_sweep_metrics_for_classification())
    model_specs = (
        _raw_classification_specs(annotation_map)
        + _exact_popdms_classification_specs(annotation_map)
        + _sae_classification_specs(sweep_df)
    )

    classification_rows = []
    fitness_frames = []
    for model_spec in model_specs:
        row = _classification_row(model_spec, annotation_map)
        fitness_df = row.pop("fitness_df")
        classification_rows.append(row)
        if not fitness_df.empty:
            fitness_frames.append(fitness_df.assign(model_label=row["model_label"]))

    pathogenicity_auc_df = pd.DataFrame(classification_rows)
    if not pathogenicity_auc_df.empty:
        pathogenicity_auc_df["method_pool"] = (
            pathogenicity_auc_df["method"].astype(str) + " / " + pathogenicity_auc_df["embedding_type"].astype(str)
        )
        pathogenicity_auc_df.to_csv(TABLE_DIR / "BRCA1_pathogenicity_auc_metrics.csv", index=False)
        if fitness_frames:
            pd.concat(fitness_frames, ignore_index=True).to_csv(
                TABLE_DIR / "BRCA1_pathogenicity_fitness_values.csv",
                index=False,
            )

        benchmark_auc_df = pathogenicity_auc_df[pathogenicity_auc_df["model_group"].eq("benchmark")].copy()
        popdms_auc_df = pathogenicity_auc_df[pathogenicity_auc_df["model_group"].eq("Regular popDMS")].copy()
        sae_auc_df = pathogenicity_auc_df[pathogenicity_auc_df["model_group"].eq("SAE sweep")].copy()
        selected_auc_rows = [benchmark_auc_df, popdms_auc_df]
        finite_sae_auc_df = sae_auc_df[np.isfinite(sae_auc_df["auc"])].copy()
        if not finite_sae_auc_df.empty:
            selected_auc_rows.extend([
                finite_sae_auc_df.loc[[finite_sae_auc_df["auc"].idxmax()]].assign(model_group="Best SAE"),
                finite_sae_auc_df.loc[[finite_sae_auc_df["auc"].idxmin()]].assign(model_group="Worst SAE"),
            ])
        selected_auc_df = pd.concat(selected_auc_rows, ignore_index=True)

        def _pathogenicity_plot_label(row):
            if row["model_group"] == "benchmark":
                return f"{row['benchmark']}\n{row['embedding_type']}"
            if row["model_group"] == "Regular popDMS":
                return "Regular popDMS"
            k_label = "NA" if pd.isna(row.get("k")) else f"{int(row['k'])}"
            n_features_label = "NA" if pd.isna(row.get("n_features")) else f"{int(row['n_features'])}"
            return f"{row['model_group']}\n{row['method']} {row['embedding_type']}\nk={k_label}, nf={n_features_label}"

        selected_auc_df["plot_label"] = selected_auc_df.apply(_pathogenicity_plot_label, axis=1)

        display(
            benchmark_auc_df.sort_values(["embedding_type", "benchmark"])[[
                "model_label",
                "benchmark",
                "embedding_type",
                "auc",
                "n_benign",
                "n_pathogenic",
                "n_variants",
            ]]
        )

        fig, ax = plt.subplots(figsize=(max(7.5, 0.60 * len(selected_auc_df)), 5.2))
        sns.barplot(
            data=selected_auc_df,
            x="plot_label",
            y="auc",
            hue="model_group",
            dodge=False,
            ax=ax,
        )
        ax.axhline(0.5, color="0.35", linestyle="--", linewidth=1)
        ax.set_ylim(0, 1)
        ax.set_xlabel("")
        ax.set_ylabel("AUC for pathogenic vs benign")
        ax.set_title("Pathogenicity classification from inferred fitness")
        ax.tick_params(axis="x", rotation=35, labelsize=7)
        for label in ax.get_xticklabels():
            label.set_ha("right")
        for container in ax.containers:
            ax.bar_label(container, fmt="%.3f", padding=2, fontsize=7)
        ax.legend(frameon=False, loc="lower right")
        fig.subplots_adjust(bottom=0.36, left=0.10, right=0.98, top=0.90)
        fig.savefig(FIGURE_DIR / "BRCA1_pathogenicity_auc_selected_models.png", dpi=FIGURE_DPI)
        plt.show()

        if not sae_auc_df.empty:
            method_order = [
                method_pool
                for method_pool in sorted(sae_auc_df["method_pool"].dropna().unique())
            ]
            fig, ax = plt.subplots(figsize=(max(7.5, 0.80 * len(method_order)), 5.2))
            sns.stripplot(
                data=sae_auc_df,
                x="method_pool",
                y="auc",
                hue="embedding_type",
                order=method_order,
                jitter=0.25,
                alpha=0.55,
                size=5,
                ax=ax,
            )
            sns.pointplot(
                data=sae_auc_df,
                x="method_pool",
                y="auc",
                order=method_order,
                estimator="mean",
                errorbar=None,
                color="black",
                markers="D",
                linestyles="none",
                ax=ax,
            )
            ax.axhline(0.5, color="0.35", linestyle="--", linewidth=1)
            for x_pos in range(len(method_order)):
                ax.axvline(x_pos, color="0.88", linewidth=0.8, zorder=0)
            ax.set_axisbelow(True)
            ax.grid(axis="y", color="0.90", linewidth=0.8)
            ax.set_ylim(0, 1)
            ax.set_xlabel("SAE type")
            ax.set_ylabel("AUC for pathogenic vs benign")
            ax.set_title("Pathogenicity AUC by SAE type")
            ax.tick_params(axis="x", rotation=25, labelsize=8)
            for label in ax.get_xticklabels():
                label.set_ha("right")
            handles, labels = ax.get_legend_handles_labels()
            if handles:
                ax.legend(handles[:sae_auc_df["embedding_type"].nunique()], labels[:sae_auc_df["embedding_type"].nunique()], frameon=False)
            fig.subplots_adjust(bottom=0.30, left=0.10, right=0.98, top=0.90)
            fig.savefig(FIGURE_DIR / "BRCA1_pathogenicity_auc_by_sae_type.png", dpi=FIGURE_DPI)
            plt.show()

        display(pathogenicity_auc_df.sort_values("auc", ascending=False).head(20))
    else:
        print("No models with available feature and inference files were found for pathogenicity classification.")


In [ ]:
PLOT_BEST_PATHOGENICITY_FITNESS_DISTRIBUTIONS = True


def _fitness_dataframe_for_auc_row(row, annotation_map):
    if row["benchmark"] == "Regular popDMS":
        selection_df = _exact_popdms_selection_coefficients(
            force_recompute=FORCE_RECOMPUTE_EXACT_POPDMS_BASELINE,
        )
        return _exact_popdms_fitness_dataframe(selection_df, annotation_map)
    seq_to_features, inference_result = _features_and_inference_for_auc_row(row)
    return _fitness_for_feature_mapping(seq_to_features, inference_result, annotation_map)


def _features_and_inference_for_auc_row(row):
    if row["model_group"] == "SAE sweep":
        return _load_pickle(row["feature_path"]), _load_pickle(row["inference_path"])
    if row["benchmark"] == "Raw embeddings":
        seq_to_features = runner.load_embeddings(REPRESENTATIVE_LAYER, row["embedding_type"])
        inference_result = runner.run_feature_inference(
            layer=REPRESENTATIVE_LAYER,
            abstraction_method="none",
            abstraction_params={"norm_scheme": NORM_SCHEME},
            embedding_type=row["embedding_type"],
        )
        return seq_to_features, inference_result
    if row["benchmark"] == "Raw embeddings - WT":
        seq_to_features, inference_result, _, _ = _raw_delta_embedding_inference(
            REPRESENTATIVE_LAYER,
            row["embedding_type"],
            NORM_SCHEME,
        )
        return seq_to_features, inference_result
    raise ValueError(f"Unsupported benchmark row: {row.to_dict()}")


if PLOT_BEST_PATHOGENICITY_FITNESS_DISTRIBUTIONS:
    if "pathogenicity_auc_df" not in globals() or pathogenicity_auc_df.empty:
        pathogenicity_auc_df = pd.read_csv(TABLE_DIR / "BRCA1_pathogenicity_auc_metrics.csv")
    annotation_map = _annotation_map_for_binary_pathogenicity()
    finite_auc_df = pathogenicity_auc_df[np.isfinite(pathogenicity_auc_df["auc"])].copy()
    best_baseline_row = finite_auc_df[finite_auc_df["model_group"].eq("benchmark")].sort_values("auc", ascending=False).iloc[0]
    best_sae_row = finite_auc_df[finite_auc_df["model_group"].eq("SAE sweep")].sort_values("auc", ascending=False).iloc[0]
    popdms_rows = finite_auc_df[finite_auc_df["model_group"].eq("Regular popDMS")].copy()
    popdms_row = popdms_rows.sort_values("auc", ascending=False).iloc[0] if not popdms_rows.empty else None

    distribution_frames = []
    distribution_specs = [
        (best_baseline_row, "Best raw baseline"),
    ]
    if popdms_row is not None:
        distribution_specs.append((popdms_row, "Regular popDMS"))
    distribution_specs.append((best_sae_row, "Best SAE"))

    for row, display_label in distribution_specs:
        try:
            fitness_df = _fitness_dataframe_for_auc_row(row, annotation_map)
        except FileNotFoundError as exc:
            print(f"Skipping {display_label} fitness distribution: {exc}")
            continue
        fitness_df = fitness_df[fitness_df["annotation"].isin(PATHOGENICITY_LABELS)].copy()
        fitness_df["model"] = display_label
        fitness_df["model_label"] = row["model_label"]
        fitness_df["auc"] = row["auc"]
        distribution_frames.append(fitness_df)

    best_pathogenicity_fitness_df = pd.concat(distribution_frames, ignore_index=True)
    best_pathogenicity_fitness_df.to_csv(
        TABLE_DIR / "BRCA1_best_pathogenicity_model_fitness_distributions.csv",
        index=False,
    )

    model_order = ["Best raw baseline", "Regular popDMS", "Best SAE"]
    output_names = {
        "Best raw baseline": "BRCA1_best_raw_baseline_pathogenicity_fitness_frequency.png",
        "Regular popDMS": "BRCA1_regular_popDMS_pathogenicity_fitness_frequency.png",
        "Best SAE": "BRCA1_best_sae_pathogenicity_fitness_frequency.png",
    }
    for model in model_order:
        model_df = best_pathogenicity_fitness_df[best_pathogenicity_fitness_df["model"].eq(model)]
        if model_df.empty:
            continue
        row = model_df.iloc[0]
        fig, ax = plt.subplots(figsize=(6.6, 4.6))
        sns.histplot(
            data=model_df,
            x="fitness",
            hue="annotation",
            hue_order=["benign", "pathogenic"],
            multiple="layer",
            bins=30,
            stat="probability",
            common_norm=False,
            alpha=0.45,
            edgecolor="white",
            linewidth=0.35,
            ax=ax,
        )
        ax.set_title(f"{model} AUC={row['auc']:.3f}")
        ax.set_xlabel("Inferred fitness")
        ax.set_ylabel("Frequency")
        ax.grid(axis="y", color="0.90", linewidth=0.8)
        ax.text(
            0.02,
            0.98,
            str(row["model_label"]),
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=8,
            bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "edgecolor": "0.85", "alpha": 0.85},
        )
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / output_names[model], dpi=FIGURE_DPI)
        plt.show()

    display(
        pathogenicity_auc_df[
            pathogenicity_auc_df["model_label"].isin([
                label for label in [
                    best_baseline_row["model_label"],
                    popdms_row["model_label"] if popdms_row is not None else None,
                    best_sae_row["model_label"],
                ]
                if label is not None
            ])
        ][[
            "model_group",
            "model_label",
            "method",
            "embedding_type",
            "auc",
            "n_benign",
            "n_pathogenic",
            "n_variants",
        ]]
    )


## MaveDB Spearman Vs ClinVar AUC

Compare each inferred model and baseline by MaveDB functional-score Spearman rho and ClinVar pathogenic-vs-benign AUC. The regular popDMS baseline uses the cached exact-popDMS selection coefficients from the cache-generation cell above, then scores variants as `1 + sum_i x_i s_i` from the inferred joint selection coefficients.

In [ ]:
RUN_MAVEDB_CLINVAR_TRADEOFF_PLOT = True
FORCE_RECOMPUTE_EXACT_POPDMS_BASELINE = False
EXACT_POPDMS_NAME = "BRCA1_exact_popDMS"
EXACT_POPDMS_GAMMA = None
EXACT_POPDMS_CORR_CUTOFF_PCT = 0.5
EXACT_POPDMS_NORM_WT = False
REGULAR_POPDMS_DIR = SEQUENCE_DIR / "regular_popdms"
EXACT_POPDMS_SELECTION_PATH = REGULAR_POPDMS_DIR / f"{EXACT_POPDMS_NAME}_selection_coefficients.csv.gz"
EXACT_POPDMS_SELECTION_TABLE_PATH = TABLE_DIR / "BRCA1_exact_popDMS_selection_coefficients.csv"
EXACT_POPDMS_FITNESS_PATH = TABLE_DIR / "BRCA1_exact_popDMS_fitness_values.csv"
SPEARMAN_AUC_TABLE_PATH = TABLE_DIR / "BRCA1_mavedb_spearman_vs_clinvar_auc.csv"
SPEARMAN_AUC_FIGURE_PATH = FIGURE_DIR / "BRCA1_mavedb_spearman_vs_clinvar_auc.png"

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import paperPop


def _exact_popdms_sequence_states():
    if runner.sequence_dataframe is None or runner.sequence_to_protein_sequence is None:
        runner.process_raw_data(drop_stop_codons=True)
    sequence_map = runner.sequence_to_protein_sequence
    mutation_site_map = runner.sequence_to_mutation_sites or {}
    wildtype_key = runner.input_data.wildtype_key
    if wildtype_key not in sequence_map:
        raise KeyError(f"Wildtype key {wildtype_key!r} is missing from sequence_to_protein_sequence.")
    wt_sequence = sequence_map[wildtype_key]
    seq_ids = sorted(runner.sequence_dataframe["SequenceIndex"].astype(str).unique())

    seq_to_states = {}
    observed_sites = set()
    for seq_id in seq_ids:
        protein_sequence = sequence_map.get(seq_id)
        if protein_sequence is None:
            continue
        states = []
        for site_idx in mutation_site_map.get(seq_id, []):
            alt_aa = protein_sequence[site_idx]
            wt_aa = wt_sequence[site_idx]
            if alt_aa != wt_aa:
                site = site_idx + 1
                states.append((site, alt_aa))
                observed_sites.add(site)
        seq_to_states[seq_id] = sorted(states, key=lambda item: item[0])

    sites = sorted(observed_sites)
    ref_aas_by_site = {site: wt_sequence[site - 1] for site in sites}
    return seq_to_states, sites, ref_aas_by_site


def _exact_popdms_frequency_paths(replicates):
    paths = []
    for rep in replicates:
        paths.extend([
            Path(paperPop.get_reads_file(REGULAR_POPDMS_DIR, EXACT_POPDMS_NAME, rep, file_ext=".csv")),
            Path(paperPop.get_aa_freq_file(REGULAR_POPDMS_DIR, EXACT_POPDMS_NAME, "single", rep)),
            Path(paperPop.get_aa_freq_file(REGULAR_POPDMS_DIR, EXACT_POPDMS_NAME, "double", rep)),
        ])
    return paths


def _write_exact_popdms_frequency_files(force_recompute=False):
    REGULAR_POPDMS_DIR.mkdir(parents=True, exist_ok=True)
    if runner.sequence_dataframe is None or runner.sequence_to_protein_sequence is None:
        runner.process_raw_data(drop_stop_codons=True)
    sequence_df = runner.sequence_dataframe.copy()
    sequence_df["SequenceIndex"] = sequence_df["SequenceIndex"].astype(str)
    sequence_df["Frequency"] = pd.to_numeric(sequence_df["Frequency"], errors="coerce").fillna(0.0)
    replicates = sorted(sequence_df["Replicate"].dropna().astype(int).unique())
    existing_paths = _exact_popdms_frequency_paths(replicates)
    if existing_paths and all(path.is_file() for path in existing_paths) and not force_recompute:
        return replicates

    seq_to_states, sites, ref_aas_by_site = _exact_popdms_sequence_states()
    if not sites:
        raise ValueError("No amino-acid substitution sites were available for exact popDMS.")
    q_states = list(paperPop.AA)

    for rep in replicates:
        rep_df = sequence_df[sequence_df["Replicate"].astype(int).eq(rep)].copy()
        generations = sorted(rep_df["Generation"].dropna().unique())
        reads_rows = []
        single_rows = []
        double_rows = []

        for generation in generations:
            time_df = rep_df[rep_df["Generation"].eq(generation)]
            total_count = float(time_df["Frequency"].sum())
            reads_rows.append({"generation": generation, "reads": total_count})
            if total_count <= 0:
                continue

            single_counts = {}
            for site in sites:
                ref_aa = ref_aas_by_site[site]
                for aa in q_states:
                    single_counts[(site, aa)] = 0.0
                single_counts[(site, ref_aa)] = total_count

            double_counts = {}
            for site_i, site_j in itertools.combinations(sites, 2):
                double_counts[(site_i, ref_aas_by_site[site_i], site_j, ref_aas_by_site[site_j])] = total_count

            for _, row in time_df.iterrows():
                count = float(row["Frequency"])
                if count <= 0:
                    continue
                states = seq_to_states.get(row["SequenceIndex"], [])
                if not states:
                    continue
                for site, aa in states:
                    single_counts[(site, aa)] += count
                for (site_i, aa_i), (site_j, aa_j) in itertools.combinations(states, 2):
                    key = (site_i, aa_i, site_j, aa_j)
                    double_counts[key] = double_counts.get(key, 0.0) + count

            for seq_i, site_i in enumerate(sites):
                ref_i = ref_aas_by_site[site_i]
                for aa_i in q_states:
                    if aa_i == ref_i:
                        continue
                    total_single = single_counts[(site_i, aa_i)]
                    for site_j in sites[:seq_i]:
                        total_double = 0.0
                        ref_j = ref_aas_by_site[site_j]
                        for aa_j in q_states:
                            if aa_j != ref_j:
                                total_double += double_counts.get((site_j, aa_j, site_i, aa_i), 0.0)
                        double_counts[(site_j, ref_j, site_i, aa_i)] = total_single - total_double
                    for site_j in sites[seq_i + 1:]:
                        total_double = 0.0
                        ref_j = ref_aas_by_site[site_j]
                        for aa_j in q_states:
                            if aa_j != ref_j:
                                total_double += double_counts.get((site_i, aa_i, site_j, aa_j), 0.0)
                        double_counts[(site_i, aa_i, site_j, ref_j)] = total_single - total_double

            for (site, aa), count in list(single_counts.items()):
                if aa != ref_aas_by_site[site]:
                    single_counts[(site, ref_aas_by_site[site])] -= count

            for (site_i, aa_i, site_j, aa_j), count in list(double_counts.items()):
                if aa_i != ref_aas_by_site[site_i] or aa_j != ref_aas_by_site[site_j]:
                    wt_key = (site_i, ref_aas_by_site[site_i], site_j, ref_aas_by_site[site_j])
                    double_counts[wt_key] = double_counts.get(wt_key, 0.0) - count

            for (site, aa), count in single_counts.items():
                if count > 0:
                    single_rows.append({
                        "generation": generation,
                        "site": site,
                        "aa": aa,
                        "frequency": count / total_count,
                        "WT_indicator": aa == ref_aas_by_site[site],
                    })

            for (site_i, aa_i, site_j, aa_j), count in double_counts.items():
                if count > 0:
                    double_rows.append({
                        "generation": generation,
                        "site_1": site_i,
                        "aa_1": aa_i,
                        "site_2": site_j,
                        "aa_2": aa_j,
                        "frequency": count / total_count,
                    })

        pd.DataFrame(reads_rows).to_csv(
            paperPop.get_reads_file(REGULAR_POPDMS_DIR, EXACT_POPDMS_NAME, rep, file_ext=".csv"),
            index=False,
        )
        pd.DataFrame(single_rows).to_csv(
            paperPop.get_aa_freq_file(REGULAR_POPDMS_DIR, EXACT_POPDMS_NAME, "single", rep),
            index=False,
            compression="gzip",
        )
        pd.DataFrame(double_rows).to_csv(
            paperPop.get_aa_freq_file(REGULAR_POPDMS_DIR, EXACT_POPDMS_NAME, "double", rep),
            index=False,
            compression="gzip",
        )
    return replicates


def _exact_popdms_selection_coefficients(force_recompute=False, allow_compute=False):
    needs_compute = force_recompute or not EXACT_POPDMS_SELECTION_PATH.is_file()
    if needs_compute and not allow_compute:
        raise FileNotFoundError(
            f"Cached exact popDMS selection coefficients are missing at {EXACT_POPDMS_SELECTION_PATH}. "
            "Run the exact popDMS inference cell with RUN_EXACT_POPDMS_INFERENCE=True first."
        )
    if force_recompute and EXACT_POPDMS_SELECTION_PATH.is_file():
        EXACT_POPDMS_SELECTION_PATH.unlink()
    if not EXACT_POPDMS_SELECTION_PATH.is_file():
        replicates = _write_exact_popdms_frequency_files(force_recompute=force_recompute)
        paperPop.infer_correlated(
            EXACT_POPDMS_NAME,
            n_replicates=len(replicates),
            corr_cutoff_pct=EXACT_POPDMS_CORR_CUTOFF_PCT,
            gamma=EXACT_POPDMS_GAMMA,
            norm_WT=EXACT_POPDMS_NORM_WT,
            freq_dir=str(REGULAR_POPDMS_DIR),
            output_dir=str(REGULAR_POPDMS_DIR),
            with_epistasis=False,
            plot_gamma=False,
        )
    selection_df = pd.read_csv(EXACT_POPDMS_SELECTION_PATH, compression="gzip")
    selection_df.to_csv(EXACT_POPDMS_SELECTION_TABLE_PATH, index=False)
    return selection_df


def _exact_popdms_fitness_dataframe(selection_df, annotation_map=None):
    seq_to_states, _, _ = _exact_popdms_sequence_states()
    non_wt_selection = selection_df[~selection_df["WT_indicator"].astype(bool)].copy()
    non_wt_selection["site"] = non_wt_selection["site"].astype(int)
    selection_lookup = {
        (int(row["site"]), str(row["amino_acid"])): float(row["joint"])
        for _, row in non_wt_selection.iterrows()
    }
    seq_ids = sorted(runner.sequence_dataframe["SequenceIndex"].astype(str).unique())
    rows = []
    for seq_id in seq_ids:
        s_sum = sum(selection_lookup.get((site, aa), 0.0) for site, aa in seq_to_states.get(seq_id, []))
        rows.append({"SequenceIndex": seq_id, "fitness": 1.0 + s_sum})
    fitness_df = pd.DataFrame(rows)
    if annotation_map is not None:
        fitness_df["annotation"] = fitness_df["SequenceIndex"].map(annotation_map)
    return fitness_df


def _spearman_for_fitness_dataframe(fitness_df, score_col="score"):
    if runner.scores_dataframe is None:
        runner.load_functional_scores()
    score_df = runner.scores_dataframe[["SequenceIndex", score_col]].dropna().copy()
    comparison_df = fitness_df.merge(score_df, on="SequenceIndex", how="inner")
    finite = np.isfinite(comparison_df["fitness"]) & np.isfinite(comparison_df[score_col])
    if finite.sum() < 3:
        return np.nan, int(finite.sum())
    rho = spearmanr(comparison_df.loc[finite, "fitness"], comparison_df.loc[finite, score_col]).statistic
    return float(rho), int(finite.sum())


def _exact_popdms_metrics_row(annotation_map):
    selection_df = _exact_popdms_selection_coefficients(
        force_recompute=FORCE_RECOMPUTE_EXACT_POPDMS_BASELINE,
    )
    fitness_df = _exact_popdms_fitness_dataframe(selection_df, annotation_map)
    fitness_df.to_csv(EXACT_POPDMS_FITNESS_PATH, index=False)

    spearman_rho, n_score_sequences = _spearman_for_fitness_dataframe(fitness_df)
    auc_metrics = _classification_metrics_for_fitness(fitness_df)
    return {
        "dataset": "BRCA1",
        "model_label": "Regular popDMS",
        "model_group": "Regular popDMS",
        "benchmark": "Regular popDMS",
        "method": "regular popDMS",
        "embedding_type": "amino-acid haplotype frequencies",
        "auc": auc_metrics["auc"],
        "n_benign": auc_metrics["n_benign"],
        "n_pathogenic": auc_metrics["n_pathogenic"],
        "n_variants": auc_metrics["n_variants"],
        "spearman_rho": spearman_rho,
        "n_score_sequences": n_score_sequences,
        "n_features": int(len(selection_df)),
        "k": np.nan,
        "feature_path": "",
        "inference_path": str(EXACT_POPDMS_SELECTION_PATH),
        "method_pool": "regular popDMS",
    }


def _existing_spearman_auc_metrics():
    if "pathogenicity_auc_df" in globals() and not pathogenicity_auc_df.empty:
        auc_df = pathogenicity_auc_df.copy()
    else:
        auc_path = TABLE_DIR / "BRCA1_pathogenicity_auc_metrics.csv"
        if not auc_path.is_file():
            raise FileNotFoundError(
                f"Missing {auc_path}. Run the pathogenicity classification section before this plot."
            )
        auc_df = pd.read_csv(auc_path)

    for column, default in [
        ("model_label", ""),
        ("model_group", ""),
        ("benchmark", ""),
        ("method", ""),
        ("embedding_type", ""),
        ("method_pool", ""),
        ("feature_path", ""),
        ("inference_path", ""),
        ("spearman_rho", np.nan),
        ("n_score_sequences", np.nan),
        ("n_features", np.nan),
        ("k", np.nan),
    ]:
        if column not in auc_df.columns:
            auc_df[column] = default
    auc_df["spearman_rho"] = pd.to_numeric(auc_df["spearman_rho"], errors="coerce")
    auc_df["n_score_sequences"] = pd.to_numeric(auc_df["n_score_sequences"], errors="coerce")

    raw_spearman_path = TABLE_DIR / "BRCA1_raw_embedding_spearman_benchmarks.csv"
    if raw_spearman_path.is_file():
        raw_spearman_df = pd.read_csv(raw_spearman_path).rename(columns={
            "spearman_rho": "raw_spearman_rho",
            "n_score_sequences": "raw_n_score_sequences",
        })
        auc_df = auc_df.merge(
            raw_spearman_df,
            on=["benchmark", "embedding_type"],
            how="left",
        )
        auc_df["spearman_rho"] = auc_df["spearman_rho"].where(
            np.isfinite(auc_df["spearman_rho"]),
            auc_df["raw_spearman_rho"],
        )
        auc_df["n_score_sequences"] = auc_df["n_score_sequences"].where(
            np.isfinite(auc_df["n_score_sequences"]),
            auc_df["raw_n_score_sequences"],
        )
        auc_df = auc_df.drop(columns=["raw_spearman_rho", "raw_n_score_sequences"], errors="ignore")

    sweep_metrics_path = TABLE_DIR / "BRCA1_DeltaSAE_sweep_metrics.csv"
    if sweep_metrics_path.is_file() and {"feature_path", "inference_path"}.issubset(auc_df.columns):
        sweep_spearman_df = pd.read_csv(sweep_metrics_path)[[
            "feature_path",
            "inference_path",
            "spearman_rho",
            "n_score_sequences",
        ]].rename(columns={
            "spearman_rho": "sweep_spearman_rho",
            "n_score_sequences": "sweep_n_score_sequences",
        })
        auc_df = auc_df.merge(
            sweep_spearman_df,
            on=["feature_path", "inference_path"],
            how="left",
        )
        auc_df["spearman_rho"] = auc_df["spearman_rho"].where(
            np.isfinite(auc_df["spearman_rho"]),
            auc_df["sweep_spearman_rho"],
        )
        auc_df["n_score_sequences"] = auc_df["n_score_sequences"].where(
            np.isfinite(auc_df["n_score_sequences"]),
            auc_df["sweep_n_score_sequences"],
        )
        auc_df = auc_df.drop(columns=["sweep_spearman_rho", "sweep_n_score_sequences"], errors="ignore")

    auc_df["dataset"] = "BRCA1"
    return auc_df


def _spearman_auc_plot_group(row):
    if row["benchmark"] == "Regular popDMS":
        return "Regular popDMS"
    if row["model_group"] == "benchmark":
        return "Raw ESM baseline"
    return "SAE sweep"


def _plot_mavedb_spearman_vs_clinvar_auc(comparison_df, output_path):
    plot_df = comparison_df[
        np.isfinite(comparison_df["spearman_rho"]) & np.isfinite(comparison_df["auc"])
    ].copy()
    if plot_df.empty:
        print("No finite Spearman/AUC pairs were available to plot.")
        return None

    fig, ax, legend_anchor = _square_axes_figure(ax_size=5.4, legend_width=3.9)
    method_palette = {
        "SAE": "#4c78a8",
        "DeltaSAE": "#b279a2",
        "DeltaEmbSAE": "#59a14f",
        "raw ESM": "#f58518",
        "raw ESM - WT": "#e45756",
        "regular popDMS": "#111111",
    }
    pool_markers = {"mean_pool": "o", "max_pool": "s", "unknown": "X"}
    baseline_markers = {"Raw embeddings": "D", "Raw embeddings - WT": "P", "Regular popDMS": "*"}

    sae_df = plot_df[plot_df["plot_group"].eq("SAE sweep")].copy()
    for (method, embedding_type), group_df in sae_df.groupby(["method", "embedding_type"], sort=True):
        ax.scatter(
            group_df["spearman_rho"],
            group_df["auc"],
            s=58,
            marker=pool_markers.get(str(embedding_type), "o"),
            color=method_palette.get(str(method), "0.45"),
            alpha=0.48,
            edgecolors="white",
            linewidths=0.45,
            label=f"{method} / {embedding_type}",
            zorder=2,
        )

    baseline_df = plot_df[~plot_df["plot_group"].eq("SAE sweep")].copy()
    for _, row in baseline_df.iterrows():
        is_regular = row["benchmark"] == "Regular popDMS"
        ax.scatter(
            row["spearman_rho"],
            row["auc"],
            s=260 if is_regular else 130,
            marker=baseline_markers.get(row["benchmark"], "D"),
            color=method_palette.get(row["method"], "#f58518"),
            alpha=0.98,
            edgecolors="white",
            linewidths=0.85,
            label="Regular popDMS" if is_regular else "Raw ESM baselines",
            zorder=5 if is_regular else 4,
        )
        ax.annotate(
            row["model_label"].replace("Raw embeddings - WT", "Raw - WT").replace("Raw embeddings", "Raw"),
            (row["spearman_rho"], row["auc"]),
            xytext=(7, 4),
            textcoords="offset points",
            fontsize=7,
            ha="left",
            va="bottom",
        )

    if not sae_df.empty:
        best_auc_row = sae_df.loc[sae_df["auc"].idxmax()]
        best_rho_row = sae_df.loc[sae_df["spearman_rho"].idxmax()]
        for label, row in [("Best SAE AUC", best_auc_row), ("Best SAE rho", best_rho_row)]:
            ax.annotate(
                label,
                (row["spearman_rho"], row["auc"]),
                xytext=(6, -9),
                textcoords="offset points",
                fontsize=7,
                ha="left",
                va="top",
                color="0.20",
            )

    ax.axhline(0.5, color="0.35", linestyle="--", linewidth=1.0, zorder=0)
    ax.axvline(0.0, color="0.72", linestyle=":", linewidth=1.0, zorder=0)
    x_values = plot_df["spearman_rho"].to_numpy(dtype=float)
    y_values = plot_df["auc"].to_numpy(dtype=float)
    x_pad = max(0.025, 0.08 * (np.nanmax(x_values) - np.nanmin(x_values)))
    y_pad = max(0.025, 0.08 * (np.nanmax(y_values) - np.nanmin(y_values)))
    ax.set_xlim(max(-1.0, np.nanmin(x_values) - x_pad), min(1.0, np.nanmax(x_values) + x_pad))
    ax.set_ylim(max(0.0, min(np.nanmin(y_values) - y_pad, 0.47)), min(1.0, max(np.nanmax(y_values) + y_pad, 0.53)))
    ax.set_xlabel("MaveDB functional-score Spearman rho")
    ax.set_ylabel("ClinVar pathogenic-vs-benign AUC")
    ax.set_title("Functional score agreement vs ClinVar classification")
    ax.grid(axis="both", color="0.90", linewidth=0.8)

    handles, labels = ax.get_legend_handles_labels()
    seen = set()
    unique_handles = []
    unique_labels = []
    for handle, label in zip(handles, labels):
        if label in seen:
            continue
        seen.add(label)
        unique_handles.append(handle)
        unique_labels.append(label)
    _external_legend(fig, ax, legend_anchor, unique_handles, unique_labels)
    fig.savefig(output_path, dpi=FIGURE_DPI)
    return fig


if RUN_MAVEDB_CLINVAR_TRADEOFF_PLOT:
    annotation_map = _annotation_map_for_binary_pathogenicity()
    existing_metrics_df = _existing_spearman_auc_metrics()
    if "benchmark" in existing_metrics_df.columns:
        existing_metrics_df = existing_metrics_df[
            ~existing_metrics_df["benchmark"].eq("Regular popDMS")
        ].copy()
    metric_frames = [existing_metrics_df]
    try:
        metric_frames.append(pd.DataFrame([_exact_popdms_metrics_row(annotation_map)]))
    except FileNotFoundError as exc:
        print(f"Skipping regular popDMS Spearman/AUC point: {exc}")
    spearman_auc_df = pd.concat(
        metric_frames,
        ignore_index=True,
        sort=False,
    )
    spearman_auc_df["plot_group"] = spearman_auc_df.apply(_spearman_auc_plot_group, axis=1)
    spearman_auc_df.to_csv(SPEARMAN_AUC_TABLE_PATH, index=False)

    fig = _plot_mavedb_spearman_vs_clinvar_auc(spearman_auc_df, SPEARMAN_AUC_FIGURE_PATH)
    if fig is not None:
        plt.show()

    display(
        spearman_auc_df.sort_values(["plot_group", "auc", "spearman_rho"], ascending=[True, False, False])[[
            "dataset",
            "plot_group",
            "model_label",
            "method",
            "embedding_type",
            "spearman_rho",
            "auc",
            "n_score_sequences",
            "n_benign",
            "n_pathogenic",
            "n_variants",
        ]]
    )